# S&P 500 risk/regime analysis — publication audit edition

This notebook preserves the v3 pipeline and adds publication-oriented validation. Run it from a fresh kernel, top to bottom. It fixes the shared-date leakage comparison and event-study episode logic; restores the complete multivariate data-loading cell; uses an investable total-return proxy with cash yield and cost sensitivity; adds retrospective temporal tests, block-bootstrap uncertainty, calibration, placebo/multiple-testing checks, external-market validation, provenance, and a reproducibility manifest.

Important limitation: the 1990–2026 sample has already been inspected during development. The section labeled “retrospective test” is therefore **not an untouched confirmatory test**. The final publication gate records that limitation rather than concealing it.

Required packages: `numpy`, `pandas`, `scikit-learn`, `yfinance`, `hmmlearn`, `matplotlib`, and (for FRED features) `pandas_datareader`. In Colab, install missing packages in a separate setup cell with `%pip install yfinance hmmlearn pandas_datareader matplotlib`.

In [1]:
import os, sys, json, warnings, hashlib, platform
from datetime import datetime, timezone
from importlib import metadata

!pip install hmmlearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf
from hmmlearn.hmm import GaussianHMM

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             brier_score_loss, log_loss, confusion_matrix,
                             precision_score, recall_score, f1_score)

# Surface warnings. HMM convergence is counted explicitly below.
warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=FutureWarning)

np.random.seed(42)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

OUT = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUT, exist_ok=True)
STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")

SAVED_FILES = []
def save_csv(obj, name, index=True):
    path = os.path.join(OUT, f"{name}_{STAMP}.csv")
    obj.to_csv(path, index=index)
    SAVED_FILES.append(path)
    print(f"[saved] {path}")
    return path

def save_json(obj, name):
    path = os.path.join(OUT, f"{name}_{STAMP}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
    SAVED_FILES.append(path)
    print(f"[saved] {path}")
    return path

def hr(title):
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 6.4 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Frozen analysis protocol (retrospective; register before any new confirmatory data)

In [2]:
TICKER                   = "^GSPC"
START_DATE               = "1990-01-01"
HORIZON_DAYS             = 63
BEAR_RULE_THRESHOLD      = 0.20
DRAWDOWN_EVENT_THRESHOLD = 0.10
RECOVERY_EVENT_THRESHOLD = 0.10
MIN_TRAIN_DAYS           = 1000
REFIT_EVERY              = 21
PROB_THRESHOLD           = 0.50
COST_BPS                 = 10.0

# These dates define a retrospective temporal report. They were chosen after
# the full sample had already been inspected and are not an untouched holdout.
RETROSPECTIVE_TEST_START = "2016-01-01"
SUBPERIODS = {
    "early_oos_1994_2007": ("1994-01-01", "2007-12-31"),
    "crisis_era_2008_2015": ("2008-01-01", "2015-12-31"),
    "retrospective_2016_end": (RETROSPECTIVE_TEST_START, None),
}

PRIMARY_MODEL            = "logit_riskoff_prob_purged"
PRIMARY_BENCHMARK        = "MA200"
BOOTSTRAP_REPS           = 1000
BOOTSTRAP_MEAN_BLOCK     = 63
PLACEBO_REPS             = 1000
COST_GRID_BPS            = [0, 5, 10, 25, 50]
EXTERNAL_MARKETS         = {
    "FTSE 100": "^FTSE",
    "DAX": "^GDAXI",
    "Nikkei 225": "^N225",
    "Hang Seng": "^HSI",
}

# Fill this with the true total count of specifications tried across all prior
# notebooks and manual experiments. Leaving it None correctly fails the gate.
DECLARED_TOTAL_TRIALS = None

PROTOCOL = {
    "research_question": "Do causal risk models improve 63-day risk identification and risk-managed performance over MA200?",
    "primary_model": PRIMARY_MODEL,
    "primary_benchmark": PRIMARY_BENCHMARK,
    "primary_threshold": PROB_THRESHOLD,
    "forecast_horizon_days": HORIZON_DAYS,
    "event_threshold": DRAWDOWN_EVENT_THRESHOLD,
    "minimum_training_days": MIN_TRAIN_DAYS,
    "refit_every_days": REFIT_EVERY,
    "retrospective_test_start": RETROSPECTIVE_TEST_START,
    "cost_grid_bps": COST_GRID_BPS,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "bootstrap_mean_block": BOOTSTRAP_MEAN_BLOCK,
    "external_markets": EXTERNAL_MARKETS,
    "confirmatory_status": "retrospective; full sample previously inspected",
}
PROTOCOL_HASH = hashlib.sha256(json.dumps(PROTOCOL, sort_keys=True).encode()).hexdigest()
print("Protocol SHA-256:", PROTOCOL_HASH)
save_json({"protocol": PROTOCOL, "sha256": PROTOCOL_HASH}, "analysis_protocol")


Protocol SHA-256: f9c8178ef9a7b12843c392f528b744338a993d7f537fe799fa9e626f23f67dd6
[saved] /content/outputs/analysis_protocol_20260922_052932_UTC.json


'/content/outputs/analysis_protocol_20260922_052932_UTC.json'

## Data download — STOP if it fails; never substitute synthetic data.

In [3]:
hr("DATA DOWNLOAD")
raw = yf.download(TICKER, start=START_DATE, auto_adjust=True, progress=False)
if raw is None or len(raw) == 0:
    sys.exit("FATAL: yfinance returned no data. Halting — no synthetic substitute.")
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = pd.DataFrame(index=raw.index)
df["price"]   = raw["Close"].astype(float)
df["log_ret"] = np.log(df["price"]).diff()
df = df.dropna().copy()
N_ROWS = len(df)
END_DATE = df.index.max().date()
print(f"Rows: {N_ROWS}   from {df.index.min().date()} to {END_DATE}")
assert N_ROWS > 8000, "unexpectedly small sample — investigate before trusting results"


DATA DOWNLOAD
Rows: 9246   from 1990-01-03 to 2026-09-21


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## PART 1 — ex-post labels (verbatim logic from the original notebook)

In [4]:
def regime_dating_with_confirm(price, threshold=0.20):
    """Return (regime, confirm_pos).
    regime: 1 = bull, 0 = bear, peaks/troughs backdated (identical to original).
    confirm_pos[i] = positional index at which day i's label first became knowable
      in real time. Days that kept the then-current regime are known same day
      (confirm_pos[i] = i). Days retroactively re-labelled after a backdated turning
      point are only knowable once the 20% move confirms them (confirm_pos[i] = the
      confirmation day). The final, still-open swing is marked n (never confirmed
      in-sample) so it is excluded from any real-time training set.
    This is what lets us PURGE look-ahead from the supervised training folds: the
    ex-post backdated label is still the evaluation target, but training may only
    use rows whose label was already settled at the training cutoff."""
    p = np.asarray(price, dtype=float)
    n = len(p)
    regime = np.ones(n, dtype=int)
    confirm = np.arange(n)                 # default: label known the same day
    cur = 1
    seg_start = 0
    ext_val, ext_idx = p[0], 0
    for i in range(1, n):
        if cur == 1:
            if p[i] > ext_val:
                ext_val, ext_idx = p[i], i
            elif p[i] <= ext_val * (1 - threshold):
                regime[seg_start:ext_idx + 1] = 1
                confirm[ext_idx + 1:i + 1] = i   # rows after the peak flip to bear, known only now
                cur, seg_start = 0, ext_idx + 1
                ext_val, ext_idx = p[i], i
        else:
            if p[i] < ext_val:
                ext_val, ext_idx = p[i], i
            elif p[i] >= ext_val * (1 + threshold):
                regime[seg_start:ext_idx + 1] = 0
                confirm[ext_idx + 1:i + 1] = i   # rows after the trough flip to bull, known only now
                cur, seg_start = 1, ext_idx + 1
                ext_val, ext_idx = p[i], i
    regime[seg_start:] = cur
    confirm[seg_start:] = n                 # final open swing: provisional, exclude from training
    return regime, confirm

def regime_dating(price, threshold=0.20):
    """Backwards-compatible wrapper returning only the regime array."""
    return regime_dating_with_confirm(price, threshold)[0]

_regime, _confirm = regime_dating_with_confirm(df["price"], BEAR_RULE_THRESHOLD)
df["bull_bear"] = _regime
df["risk_off"]  = 1 - df["bull_bear"]

# Positional index at which each supervised training LABEL becomes knowable.
#  - risk_off nowcast label: settled when the 20% swing confirms (backdated).
#  - target_drawdown / target_recovery forecast labels: settled HORIZON_DAYS later
#    (they look forward 63 trading days), so the last HORIZON_DAYS rows of any
#    training window are not yet knowable and must be purged.
RISKOFF_KNOWN_POS = _confirm.copy()
FWD_KNOWN_POS     = np.arange(len(df)) + HORIZON_DAYS

price = df["price"].values
n = len(price)
run_peak = np.maximum.accumulate(price)
df["drawdown"] = price / run_peak - 1.0   # causal: decline from running peak, known at t

fwd_dd = np.full(n, np.nan)
fwd_up = np.full(n, np.nan)
for t in range(n):
    end = min(n, t + 1 + HORIZON_DAYS)
    if t + 1 < end:
        seg = price[t + 1:end]
        fwd_dd[t] = seg.min() / price[t] - 1.0
        fwd_up[t] = seg.max() / price[t] - 1.0
df["fwd_min_ret"] = fwd_dd
df["fwd_max_ret"] = fwd_up
df["target_drawdown"] = (df["fwd_min_ret"] <= -DRAWDOWN_EVENT_THRESHOLD).astype(float)
df["target_recovery"] = (((df["fwd_max_ret"] >= RECOVERY_EVENT_THRESHOLD) &
                          (df["risk_off"] == 1))).astype(float)
tail = df.index[-HORIZON_DAYS:]
df.loc[tail, ["target_drawdown", "target_recovery"]] = np.nan

RISKOFF_SHARE = df["risk_off"].mean()
DD_BASE_RATE  = df["target_drawdown"].mean()
REC_BASE_RATE = df.loc[df.risk_off == 1, "target_recovery"].mean()
hr("EX-POST LABELS")
print(f"Share of days risk-off (ex-post):        {RISKOFF_SHARE:.4f}")
print(f"Base rate P(major drawdown next 3m):     {DD_BASE_RATE:.4f}")
print(f"Base rate P(recovery | bear, next 3m):   {REC_BASE_RATE:.4f}")
# Sanity: base rates should be near prior-run values (0.115 / 0.139).
assert abs(RISKOFF_SHARE - 0.115) < 0.03, f"risk_off base rate {RISKOFF_SHARE} far from prior 0.115"
assert abs(DD_BASE_RATE - 0.139) < 0.03, f"drawdown base rate {DD_BASE_RATE} far from prior 0.139"

# Smoothed full-sample HMM — EX-POST upper-bound reference only.
_full = GaussianHMM(n_components=2, covariance_type="full", n_iter=300, random_state=42)
_full.fit(df["log_ret"].values.reshape(-1, 1))
_smooth = _full.predict_proba(df["log_ret"].values.reshape(-1, 1))
_riskoff_state = int(np.argmin(_full.means_[:, 0]))
df["hmm_smoothed_riskoff"] = _smooth[:, _riskoff_state]

# ----------------------------------------------------------------------------
# PART 2 — real-time walk-forward HMM (single pass: filtered prob daily +
# stored (alpha, params) on the monthly grid for the MC drawdown forecast).
# Methodologically identical to the original two-pass version.
# ----------------------------------------------------------------------------
CKPT = os.path.join(OUT, "model_checkpoint_v4.pkl")
USE_CKPT = os.environ.get("USE_CHECKPOINT") == "1" and os.path.exists(CKPT)

def canonicalize(model):
    """Order states so index 0 = risk-on (high mean), 1 = risk-off (low mean)."""
    mu = model.means_[:, 0]
    order = np.argsort(mu)[::-1]
    mu = mu[order]
    var = model.covars_[:, 0, 0][order]
    trans = model.transmat_[order][:, order]
    start = model.startprob_[order]
    return mu, var, trans, start

hr("WALK-FORWARD HMM (filtered, expanding window)")
if USE_CKPT:
    # Reload model outputs computed earlier TODAY by this same script from real
    # data (integrity: this is a resume mechanism, not a data substitute).
    ck = pd.read_pickle(CKPT)
    if str(ck["end_date"]) != str(END_DATE) or ck["n_rows"] != N_ROWS:
        sys.exit("FATAL: checkpoint does not match freshly downloaded data — rerun without USE_CHECKPOINT.")
    for c in ck["model_cols"].columns:
        df[c] = ck["model_cols"][c]
    mc_series = ck["mc_series"]
    nonconverged, n_fits = ck["nonconverged"], ck["n_fits"]
    print(f"[checkpoint] restored model columns from {CKPT} (computed {ck['stamp']})")
returns = df["log_ret"].values
N = len(returns)
if not USE_CKPT:
    filt_riskoff = np.full(N, np.nan)
    grid_states = []          # (t, alpha, mu, var, trans) at monthly grid points
    nonconverged = 0
    n_fits = 0

mu = var = trans = None
alpha = None
started = False
refit_points = set(range(MIN_TRAIN_DAYS, N, REFIT_EVERY))
for t in (range(N) if not USE_CKPT else []):
    if t < MIN_TRAIN_DAYS:
        continue
    if (t == MIN_TRAIN_DAYS) or (t in refit_points):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")   # counted via monitor_ instead of spamming
            m = GaussianHMM(n_components=2, covariance_type="full",
                            n_iter=200, random_state=42)
            m.fit(returns[:t].reshape(-1, 1))
        n_fits += 1
        if not m.monitor_.converged:
            nonconverged += 1
        mu, var, trans, start = canonicalize(m)
        if not started:
            alpha = start.copy()
    x = returns[t]
    b = (1.0 / np.sqrt(2 * np.pi * var)) * np.exp(-0.5 * (x - mu) ** 2 / var)
    if not started:
        alpha = start * b
        started = True
    else:
        alpha = (alpha @ trans) * b
    s = alpha.sum()
    alpha = alpha / s if (s > 0 and np.isfinite(s)) else np.array([0.5, 0.5])
    filt_riskoff[t] = alpha[1]
    if t % REFIT_EVERY == 0:
        grid_states.append((t, alpha.copy(), mu.copy(), var.copy(), trans.copy()))

if not USE_CKPT:
    df["hmm_filtered_riskoff"] = filt_riskoff
print(f"HMM refits: {n_fits}; non-converged fits (n_iter=200): {nonconverged} "
      f"({(nonconverged / n_fits if n_fits else 0):.1%}) — expected with short expanding windows; reported, not hidden.")
assert df["hmm_filtered_riskoff"].dropna().between(0, 1).all()

# MC drawdown forecast from stored grid states. Uses only info up to t (causal).
def simulate_drawdown_prob(alpha0, mu, var, trans, horizon, thresh, rng, n_paths=400):
    sd = np.sqrt(var)
    states = np.where(rng.random(n_paths) < alpha0[1], 1, 0)
    logprice = np.zeros(n_paths)
    worst = np.zeros(n_paths)
    for _ in range(horizon):
        r = rng.normal(mu[states], sd[states])
        logprice += r
        worst = np.minimum(worst, logprice)
        p_to1 = trans[states, 1]
        states = np.where(rng.random(n_paths) < p_to1, 1, 0)
    return np.mean((np.exp(worst) - 1.0) <= -thresh)

hr("HMM MONTE-CARLO DRAWDOWN FORECAST (seeds 42 and 7 for sensitivity)")
if not USE_CKPT:
    mc_series = {}
for seed in ((42, 7) if not USE_CKPT else ()):
    rng = np.random.default_rng(seed)
    s = pd.Series(np.nan, index=df.index)
    for (t, a, mu_, var_, tr_) in grid_states:
        s.iloc[t] = simulate_drawdown_prob(a, mu_, var_, tr_, HORIZON_DAYS,
                                           DRAWDOWN_EVENT_THRESHOLD, rng)
    mc_series[seed] = s.ffill()
if not USE_CKPT:
    df["hmm_drawdown_prob"] = mc_series[42]   # canonical (seed 42)
assert df["hmm_drawdown_prob"].dropna().between(0, 1).all()

# ----------------------------------------------------------------------------
# Supervised walk-forward models (verbatim features/logic from original).
# All features are trailing-window transforms of price/returns → causal.
# ----------------------------------------------------------------------------
hr("SUPERVISED WALK-FORWARD MODELS")
feat = pd.DataFrame(index=df.index)
r = df["log_ret"]
feat["ret_21"]  = r.rolling(21).sum()
feat["ret_63"]  = r.rolling(63).sum()
feat["ret_126"] = r.rolling(126).sum()
feat["ret_252"] = r.rolling(252).sum()
feat["vol_21"]  = r.rolling(21).std() * np.sqrt(252)
feat["vol_63"]  = r.rolling(63).std() * np.sqrt(252)
feat["mom_50_200"] = (df["price"].rolling(50).mean() /
                      df["price"].rolling(200).mean() - 1.0)
feat["dd_from_252high"] = df["price"] / df["price"].rolling(252).max() - 1.0
FEATURES = list(feat.columns)
df[FEATURES] = feat

def make_logit():
    return Pipeline([("sc", StandardScaler()),
                     ("lr", LogisticRegression(max_iter=1000, C=1.0))])

def make_gbm():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                          max_iter=300, random_state=42)

def walk_forward_predict(Xdf, y, make_model, min_train, step, known_pos=None):
    """Expanding-window walk-forward OOS probabilities. No look-ahead.

    known_pos[i] = positional index at which label y[i] first becomes knowable.
    When training with cutoff t (predicting rows t:end), only rows whose label was
    settled strictly before t are used (PURGE). Passing known_pos=None reproduces
    the original, un-purged behaviour and is kept ONLY for reconciliation."""
    idx = Xdf.index
    n = len(Xdf)
    if known_pos is None:
        known_pos = np.arange(n)          # label known same day == original (leaky) behaviour
    known_pos = np.asarray(known_pos)
    proba = pd.Series(np.nan, index=idx)
    t = min_train
    while t < n:
        end = min(n, t + step)
        settled = known_pos[:t] < t        # purge labels not yet knowable at cutoff t
        Xtr = Xdf.iloc[:t][settled]
        ytr = y.iloc[:t][settled]
        m_tr = ytr.notna() & Xtr.notna().all(axis=1)
        if m_tr.sum() > 100 and ytr[m_tr].nunique() > 1:
            model = make_model()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model.fit(Xtr[m_tr], ytr[m_tr])
            Xte = Xdf.iloc[t:end]
            m_te = Xte.notna().all(axis=1)
            if m_te.any():
                p = model.predict_proba(Xte[m_te])[:, 1]
                proba.loc[Xte.index[m_te]] = p
        t = end
    return proba

Xdf = df[FEATURES]
if not USE_CKPT:
    # LEAKY versions (no purge) — kept ONLY so the Phase-1 reconciliation still
    # reproduces the original pre-fix numbers and proves they were real.
    df["logit_drawdown_prob"] = walk_forward_predict(Xdf, df["target_drawdown"], make_logit, MIN_TRAIN_DAYS, REFIT_EVERY)
    df["gbm_drawdown_prob"]   = walk_forward_predict(Xdf, df["target_drawdown"], make_gbm, MIN_TRAIN_DAYS, REFIT_EVERY)
    df["logit_riskoff_prob"]  = walk_forward_predict(Xdf, df["risk_off"].astype(float), make_logit, MIN_TRAIN_DAYS, REFIT_EVERY)
    # PURGED / leakage-corrected versions (PRIMARY going forward). Training folds
    # exclude labels not yet knowable at the cutoff (see RISKOFF_KNOWN_POS / FWD_KNOWN_POS).
    df["logit_drawdown_prob_purged"] = walk_forward_predict(Xdf, df["target_drawdown"], make_logit, MIN_TRAIN_DAYS, REFIT_EVERY, known_pos=FWD_KNOWN_POS)
    df["gbm_drawdown_prob_purged"]   = walk_forward_predict(Xdf, df["target_drawdown"], make_gbm, MIN_TRAIN_DAYS, REFIT_EVERY, known_pos=FWD_KNOWN_POS)
    df["logit_riskoff_prob_purged"]  = walk_forward_predict(Xdf, df["risk_off"].astype(float), make_logit, MIN_TRAIN_DAYS, REFIT_EVERY, known_pos=RISKOFF_KNOWN_POS)
for c in ["logit_drawdown_prob", "gbm_drawdown_prob", "logit_riskoff_prob",
          "logit_drawdown_prob_purged", "gbm_drawdown_prob_purged", "logit_riskoff_prob_purged"]:
    assert df[c].dropna().between(0, 1).all(), c
print("Supervised walk-forward probabilities computed (leaky + leakage-corrected).")

MODEL_COLS = ["hmm_filtered_riskoff", "hmm_drawdown_prob",
              "logit_drawdown_prob", "gbm_drawdown_prob", "logit_riskoff_prob",
              "logit_drawdown_prob_purged", "gbm_drawdown_prob_purged", "logit_riskoff_prob_purged"]
if not USE_CKPT:
    pd.to_pickle({"model_cols": df[MODEL_COLS], "mc_series": mc_series,
                  "nonconverged": nonconverged, "n_fits": n_fits,
                  "end_date": END_DATE, "n_rows": N_ROWS, "stamp": STAMP}, CKPT)
    print(f"[checkpoint] saved model columns to {CKPT}")


EX-POST LABELS
Share of days risk-off (ex-post):        0.1139
Base rate P(major drawdown next 3m):     0.1375
Base rate P(recovery | bear, next 3m):   0.1130


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



WALK-FORWARD HMM (filtered, expanding window)


HMM refits: 393; non-converged fits (n_iter=200): 0 (0.0%) — expected with short expanding windows; reported, not hidden.

HMM MONTE-CARLO DRAWDOWN FORECAST (seeds 42 and 7 for sensitivity)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



SUPERVISED WALK-FORWARD MODELS
Supervised walk-forward probabilities computed (leaky + leakage-corrected).
[checkpoint] saved model columns to /content/outputs/model_checkpoint_v4.pkl


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Evaluation helpers (verbatim) + shared strategy machinery with costs

In [5]:
def evaluate(prob, target, name, thr=PROB_THRESHOLD):
    m = prob.notna() & target.notna()
    p, y = prob[m].values, target[m].values.astype(int)
    if len(np.unique(y)) < 2:
        return None
    yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    assert tn + fp + fn + tp == len(y)
    return {"model": name, "n": len(y), "base_rate": y.mean(),
            "ROC_AUC": roc_auc_score(y, p),
            "PR_AUC": average_precision_score(y, p),
            "Brier": brier_score_loss(y, p),
            "LogLoss": log_loss(y, p, labels=[0, 1]),
            "Precision": precision_score(y, yhat, zero_division=0),
            "Recall": recall_score(y, yhat, zero_division=0),
            "F1": f1_score(y, yhat, zero_division=0),
            "FP_rate": fp / (fp + tn) if (fp + tn) else np.nan,
            "TP": tp, "FP": fp, "FN": fn, "TN": tn}

mkt_ret = df["price"].pct_change()

def strategy_stats(daily_ret):
    daily_ret = daily_ret.dropna()
    eq = (1 + daily_ret).cumprod()
    yrs = (eq.index[-1] - eq.index[0]).days / 365.25
    cagr = eq.iloc[-1] ** (1 / yrs) - 1
    vol = daily_ret.std() * np.sqrt(252)
    sharpe = (daily_ret.mean() * 252) / vol if vol else np.nan
    mdd = (eq / eq.cummax() - 1).min()
    return eq, {"CAGR": cagr, "AnnVol": vol, "Sharpe": sharpe, "MaxDD": mdd}

def run_strategy(position, mask, cost_bps=COST_BPS):
    """position in [0,1] decided at close of t, effective t+1 (lag = no look-ahead).
    Cost: cost_bps * |change in effective position| charged on the trade day."""
    pos_eff = position.shift(1)
    gross = (pos_eff * mkt_ret)[mask]
    trades = pos_eff.diff().abs()[mask].fillna(0.0)
    net = gross - (cost_bps / 1e4) * trades
    _, g = strategy_stats(gross)
    _, nt = strategy_stats(net)
    turnover = float(trades.sum())
    return g, nt, turnover

def mean_run_length(sig):
    """Average length (trading days) of consecutive signal==1 runs."""
    sig = sig.dropna().astype(int)
    grp = (sig != sig.shift()).cumsum()
    runs = sig.groupby(grp).agg(["first", "size"])
    on = runs[runs["first"] == 1]["size"]
    return float(on.mean()) if len(on) else np.nan

## PHASE 1 — metrics + reconciliation against prior run (2026-06-12, 9178 rows)

In [6]:
hr("PHASE 1 — CLASSIFICATION & CALIBRATION (fresh run)")
rows = [
    evaluate(df["hmm_filtered_riskoff"], df["risk_off"].astype(float), "HMM filtered (risk-off nowcast)"),
    evaluate(df["logit_riskoff_prob"], df["risk_off"].astype(float), "Logit (risk-off nowcast)"),
    evaluate(df["hmm_drawdown_prob"], df["target_drawdown"], "HMM sim (drawdown forecast)"),
    evaluate(df["logit_drawdown_prob"], df["target_drawdown"], "Logit (drawdown forecast)"),
    evaluate(df["gbm_drawdown_prob"], df["target_drawdown"], "GBM (drawdown forecast)"),
    evaluate(df["hmm_smoothed_riskoff"], df["risk_off"].astype(float), "HMM smoothed [EX-POST, leaks future]"),
]
metrics_tbl = pd.DataFrame([x for x in rows if x]).set_index("model")
print(metrics_tbl.round(4).to_string())
save_csv(metrics_tbl.round(6), "metrics_classification")

# seed sensitivity of the MC drawdown forecast
auc_mc = {}
for seed, s in mc_series.items():
    m = s.notna() & df["target_drawdown"].notna()
    auc_mc[seed] = roc_auc_score(df["target_drawdown"][m].astype(int), s[m])
print(f"\nMC drawdown-forecast ROC-AUC by seed: " +
      ", ".join(f"seed {k}: {v:.4f}" for k, v in auc_mc.items()) +
      f"  (spread {abs(auc_mc[42]-auc_mc[7]):.4f})")

# false alarms / whipsaws
def false_alarm_stats(prob, thr=PROB_THRESHOLD):
    m = prob.notna()
    sig = (prob[m] >= thr).astype(int)
    bull = (df.loc[m, "risk_off"] == 0)
    fpr_bull = sig[bull].mean()
    flips = int((sig.diff().abs() == 1).sum())
    on_episodes = int(((sig.diff() == 1)).sum())
    return {"FPR_in_bull": fpr_bull, "signal_flips": flips, "risk_off_episodes": on_episodes}

fa = pd.DataFrame({
    "HMM filtered": false_alarm_stats(df["hmm_filtered_riskoff"]),
    "Logit risk-off": false_alarm_stats(df["logit_riskoff_prob"]),
}).T
hr("PHASE 1 — FALSE ALARMS / WHIPSAWS")
print(fa.round(4).to_string())
save_csv(fa.round(6), "metrics_false_alarms")

# original toy strategy (full cash, no costs) for reconciliation
signal = (df["hmm_filtered_riskoff"] < PROB_THRESHOLD).astype(float).shift(1)
mask0 = signal.notna() & df["hmm_filtered_riskoff"].notna()
_, s_stats = strategy_stats((signal * mkt_ret)[mask0])
_, b_stats = strategy_stats(mkt_ret[mask0])
strat_tbl = pd.DataFrame({"Risk-managed (HMM)": s_stats, "Buy & hold": b_stats}).T
hr("PHASE 1 — ORIGINAL STRATEGY vs BUY-AND-HOLD (no costs, for reconciliation)")
print(strat_tbl.round(4).to_string())
save_csv(strat_tbl.round(6), "metrics_strategy_original")

# event delays, original style, for reconciliation
CRISES = {"2008 GFC": ("2007-06-01", "2009-12-31"),
          "2020 COVID": ("2020-01-01", "2020-12-31"),
          "2022": ("2021-11-01", "2023-06-30")}

def regime_onsets_in(window):
    sub = df.loc[window[0]:window[1]]
    b = sub["risk_off"].values
    idx = sub.index
    segs, s = [], None
    for i in range(len(b)):
        if b[i] == 1 and s is None:
            s = i
        if b[i] == 0 and s is not None:
            segs.append((idx[s], idx[i - 1])); s = None
    if s is not None:
        segs.append((idx[s], idx[-1]))
    return segs

def delay_days(prob, onset, end, thr=PROB_THRESHOLD):
    sig = (prob >= thr)
    win_lo = onset - pd.Timedelta(days=120)
    cand = sig.loc[win_lo:onset + pd.Timedelta(days=120)]
    crossed = cand.index[cand.values]
    det = (crossed[0] - onset).days if len(crossed) else np.nan
    post = (prob.loc[end:] < thr)
    exited = post.index[post.values]
    ext = (exited[0] - end).days if len(exited) else np.nan
    first_warn = crossed[0] if len(crossed) else pd.NaT
    exit_date = exited[0] if len(exited) else pd.NaT
    return det, ext, first_warn, exit_date

ev_rows = []
for name, win in CRISES.items():
    for onset, end in regime_onsets_in(win):
        det, ext, _, _ = delay_days(df["hmm_filtered_riskoff"], onset, end)
        ev_rows.append({"crisis": name, "onset": onset.date(), "end": end.date(),
                        "detection_delay_days": det, "exit_delay_days": ext})
event_tbl = pd.DataFrame(ev_rows)
hr("PHASE 1 — EVENT DELAYS, ORIGINAL STYLE (HMM filtered)")
print(event_tbl.to_string(index=False))
save_csv(event_tbl, "metrics_event_delays_original", index=False)

# --- reconciliation table -----------------------------------------------------
PRIOR = {
    "rows": 9178, "end_date": "2026-06-12",
    "riskoff_share": 0.115, "dd_base_rate": 0.139, "recovery_base_rate": 0.113,
    "hmm_filt_auc": 0.848, "hmm_filt_prauc": 0.387, "hmm_filt_brier": 0.172,
    "hmm_filt_precision": 0.343, "hmm_filt_recall": 0.748, "hmm_filt_fpr": 0.212,
    "logit_riskoff_auc": 0.814, "logit_riskoff_prauc": 0.478,
    "logit_riskoff_precision": 0.585, "logit_riskoff_recall": 0.450, "logit_riskoff_fpr": 0.060,
    "hmm_sim_auc": 0.654, "logit_dd_auc": 0.690, "gbm_dd_auc": 0.729,
    "hmm_smoothed_auc": 0.852,
    "fa_hmm_fpr_bull": 0.212, "fa_hmm_flips": 413, "fa_logit_fpr_bull": 0.060, "fa_logit_flips": 116,
    "strat_cagr": 0.0575, "strat_sharpe": 0.570, "strat_maxdd": -0.426,
    "bh_cagr": 0.0892, "bh_sharpe": 0.554, "bh_maxdd": -0.568,
}
g = lambda model, col: float(metrics_tbl.loc[model, col])
NEW = {
    "rows": N_ROWS, "end_date": str(END_DATE),
    "riskoff_share": RISKOFF_SHARE, "dd_base_rate": DD_BASE_RATE, "recovery_base_rate": REC_BASE_RATE,
    "hmm_filt_auc": g("HMM filtered (risk-off nowcast)", "ROC_AUC"),
    "hmm_filt_prauc": g("HMM filtered (risk-off nowcast)", "PR_AUC"),
    "hmm_filt_brier": g("HMM filtered (risk-off nowcast)", "Brier"),
    "hmm_filt_precision": g("HMM filtered (risk-off nowcast)", "Precision"),
    "hmm_filt_recall": g("HMM filtered (risk-off nowcast)", "Recall"),
    "hmm_filt_fpr": g("HMM filtered (risk-off nowcast)", "FP_rate"),
    "logit_riskoff_auc": g("Logit (risk-off nowcast)", "ROC_AUC"),
    "logit_riskoff_prauc": g("Logit (risk-off nowcast)", "PR_AUC"),
    "logit_riskoff_precision": g("Logit (risk-off nowcast)", "Precision"),
    "logit_riskoff_recall": g("Logit (risk-off nowcast)", "Recall"),
    "logit_riskoff_fpr": g("Logit (risk-off nowcast)", "FP_rate"),
    "hmm_sim_auc": g("HMM sim (drawdown forecast)", "ROC_AUC"),
    "logit_dd_auc": g("Logit (drawdown forecast)", "ROC_AUC"),
    "gbm_dd_auc": g("GBM (drawdown forecast)", "ROC_AUC"),
    "hmm_smoothed_auc": g("HMM smoothed [EX-POST, leaks future]", "ROC_AUC"),
    "fa_hmm_fpr_bull": float(fa.loc["HMM filtered", "FPR_in_bull"]),
    "fa_hmm_flips": float(fa.loc["HMM filtered", "signal_flips"]),
    "fa_logit_fpr_bull": float(fa.loc["Logit risk-off", "FPR_in_bull"]),
    "fa_logit_flips": float(fa.loc["Logit risk-off", "signal_flips"]),
    "strat_cagr": s_stats["CAGR"], "strat_sharpe": s_stats["Sharpe"], "strat_maxdd": s_stats["MaxDD"],
    "bh_cagr": b_stats["CAGR"], "bh_sharpe": b_stats["Sharpe"], "bh_maxdd": b_stats["MaxDD"],
}
EXPLAIN = {k: f"sample extended to {END_DATE}" for k in PRIOR}
EXPLAIN["hmm_sim_auc"] = ("sample extended + Monte-Carlo RNG stream differs from prior run "
                          "(prior used the global np.random stream); seed-sensitivity reported above")
recon = pd.DataFrame({
    "metric": list(PRIOR.keys()),
    "prior": [PRIOR[k] for k in PRIOR],
    "new": [NEW[k] for k in PRIOR],
})
recon["abs_diff"] = [
    (abs(float(p) - float(v)) if isinstance(p, (int, float)) and isinstance(v, (int, float)) else np.nan)
    for p, v in zip(recon["prior"], recon["new"])]
recon["explanation"] = [EXPLAIN[k] for k in PRIOR]
hr("PHASE 1 — RECONCILIATION vs PRIOR RUN")
print(recon.to_string(index=False))
save_csv(recon, "reconciliation_phase1", index=False)

qual_checks = {
    "nowcast_auc_above_0p8": bool(NEW["hmm_filt_auc"] > 0.8 and NEW["logit_riskoff_auc"] > 0.8),
    "forecast_auc_well_below_nowcast": bool(max(NEW["hmm_sim_auc"], NEW["logit_dd_auc"], NEW["gbm_dd_auc"])
                                            < min(NEW["hmm_filt_auc"], NEW["logit_riskoff_auc"])),
    "hmm_whipsaws_heavily_vs_logit": bool(NEW["fa_hmm_flips"] > 2 * NEW["fa_logit_flips"]),
    "strategy_trades_cagr_for_lower_maxdd": bool((NEW["strat_cagr"] < NEW["bh_cagr"]) and
                                                 (NEW["strat_maxdd"] > NEW["bh_maxdd"])),
}
print("\nQualitative story checks:", json.dumps(qual_checks, indent=2))


PHASE 1 — CLASSIFICATION & CALIBRATION (fresh run)
                                         n  base_rate  ROC_AUC  PR_AUC  Brier  LogLoss  Precision  Recall     F1  FP_rate   TP    FP    FN    TN
model                                                                                                                                           
HMM filtered (risk-off nowcast)       8246     0.1277   0.8484  0.3861 0.1708   0.7691     0.3417  0.7483 0.4692   0.2110  788  1518   265  5675
Logit (risk-off nowcast)              6650     0.1567   0.8151  0.4778 0.1072   0.4189     0.5848  0.4501 0.5087   0.0594  469   333   573  5275
HMM sim (drawdown forecast)           8175     0.1469   0.6375  0.2104 0.1228   0.4050     0.0000  0.0000 0.0000   0.0000    0     0  1201  6974
Logit (drawdown forecast)             7217     0.1663   0.6888  0.3522 0.1408   0.4830     0.3942  0.1708 0.2384   0.0524  205   315   995  5702
GBM (drawdown forecast)               7217     0.1663   0.7280  0.3822 0.1350 

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[saved] /content/outputs/metrics_classification_20260922_052932_UTC.csv

MC drawdown-forecast ROC-AUC by seed: seed 42: 0.6375, seed 7: 0.6422  (spread 0.0047)

PHASE 1 — FALSE ALARMS / WHIPSAWS
                FPR_in_bull  signal_flips  risk_off_episodes
HMM filtered         0.2110      416.0000           208.0000
Logit risk-off       0.0594      116.0000            58.0000
[saved] /content/outputs/metrics_false_alarms_20260922_052932_UTC.csv

PHASE 1 — ORIGINAL STRATEGY vs BUY-AND-HOLD (no costs, for reconciliation)
                     CAGR  AnnVol  Sharpe   MaxDD
Risk-managed (HMM) 0.0588  0.1085  0.5815 -0.4257
Buy & hold         0.0899  0.1852  0.5581 -0.5678
[saved] /content/outputs/metrics_strategy_original_20260922_052932_UTC.csv

PHASE 1 — EVENT DELAYS, ORIGINAL STYLE (HMM filtered)
    crisis      onset        end  detection_delay_days  exit_delay_days
  2008 GFC 2007-10-10 2008-11-20                   -76              201
  2008 GFC 2009-01-07 2009-03-09                  -1

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## TASK A — precise target definitions + computed base rates

In [7]:
hr("TASK A — TARGET DEFINITIONS (computed base rates)")
bear_segs_all = regime_onsets_in((df.index[0], df.index[-1]))
n_bear_episodes = len(bear_segs_all)
# peak vs onset coincidence check (by construction onset = trading day after peak)
peak_onset_gaps = []
for onset, end in bear_segs_all:
    pk = df.loc[:onset, "price"].idxmax()
    peak_onset_gaps.append((onset - pk).days)
taskA = pd.DataFrame({
    "definition": [
        f"risk_off day: day inside a bear episode under the {BEAR_RULE_THRESHOLD:.0%} rule, "
        "bear backdated to start the trading day AFTER the price peak, ending at the trough",
        f"target_drawdown: 1 if price falls > {DRAWDOWN_EVENT_THRESHOLD:.0%} at any point in the "
        f"next {HORIZON_DAYS} trading days (label uses future data; last {HORIZON_DAYS} rows NaN)",
        f"target_recovery: 1 if in a bear at t AND price rises > {RECOVERY_EVENT_THRESHOLD:.0%} "
        f"within the next {HORIZON_DAYS} trading days",
        "event onset: backdated bear-episode start (detection delay measured against this)",
        "nowcast FP: risk-off call on an ex-post bull day; FN: no call on an ex-post bear day",
        "forecast FP: P(drawdown)>=thr but no >10% fall in next 63 days; FN: fall occurs, no signal",
    ],
    "computed_value": [
        f"share risk_off = {RISKOFF_SHARE:.4f} over {N_ROWS} days; {n_bear_episodes} bear episodes",
        f"base rate = {DD_BASE_RATE:.4f}",
        f"base rate (given bear) = {REC_BASE_RATE:.4f}",
        f"median gap peak->onset = {np.median(peak_onset_gaps):.0f} calendar days "
        "(coincide by construction: onset is the next trading day after the peak)",
        f"see confusion cells in metrics_classification CSV (nowcast rows)",
        f"see confusion cells in metrics_classification CSV (forecast rows)",
    ],
})
print(taskA.to_string(index=False))
save_csv(taskA, "taskA_definitions_base_rates", index=False)


TASK A — TARGET DEFINITIONS (computed base rates)
                                                                                                                                    definition                                                                                                    computed_value
risk_off day: day inside a bear episode under the 20% rule, bear backdated to start the trading day AFTER the price peak, ending at the trough                                                           share risk_off = 0.1139 over 9246 days; 6 bear episodes
                   target_drawdown: 1 if price falls > 10% at any point in the next 63 trading days (label uses future data; last 63 rows NaN)                                                                                                base rate = 0.1375
                                                    target_recovery: 1 if in a bear at t AND price rises > 10% within the next 63 trading days                                    

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'/content/outputs/taskA_definitions_base_rates_20260922_052932_UTC.csv'

## TASK B — expanded event-study table

In [8]:
hr("TASK B — EPISODE-SPECIFIC EVENT STUDY")
NOWCASTS = {
    "HMM filtered": df["hmm_filtered_riskoff"],
    "Logit risk-off": df["logit_riskoff_prob_purged"],
}

def episode_signal_dates(prob, peak, onset, end, thr=PROB_THRESHOLD):
    '''Require a fresh off-to-on signal transition for each distinct episode.

    This prevents a warning already active during an earlier bear leg from being
    credited as an early warning for a later episode. The search begins 120
    calendar days before the episode-specific peak and ends 120 days after onset.
    '''
    sig = (prob >= thr).fillna(False)
    fresh_cross = sig & ~sig.shift(1, fill_value=False)
    lo = peak - pd.Timedelta(days=120)
    hi = onset + pd.Timedelta(days=120)
    candidates = fresh_cross.loc[lo:hi]
    crossed = candidates.index[candidates.values]
    first_warning = crossed[0] if len(crossed) else pd.NaT

    post = (~sig).loc[end:]
    exits = post.index[post.values]
    exit_date = exits[0] if len(exits) else pd.NaT
    return first_warning, exit_date

taskB_rows, verdicts = [], []
for cname, win in CRISES.items():
    for episode_number, (onset, end) in enumerate(regime_onsets_in(win), start=1):
        onset_pos = df.index.get_loc(onset)
        peak = df.index[onset_pos - 1]  # regime dating starts the bear after its own peak
        episode = df.loc[onset:end]
        bottom = episode["price"].idxmin()
        for mname, prob in NOWCASTS.items():
            first_warn, exit_date = episode_signal_dates(prob, peak, onset, end)
            delay = (first_warn - onset).days if pd.notna(first_warn) else np.nan
            dd_at_warn = float(df.loc[first_warn, "drawdown"]) if pd.notna(first_warn) else np.nan
            exit_lag = (exit_date - bottom).days if pd.notna(exit_date) else np.nan
            taskB_rows.append({
                "crisis": cname,
                "episode": episode_number,
                "model": mname,
                "episode_peak_date": peak.date(),
                "drawdown_onset_date": onset.date(),
                "bottom_date": bottom.date(),
                "fresh_warning_date": first_warn.date() if pd.notna(first_warn) else None,
                "lead_lag_vs_onset_days": delay,
                "drawdown_at_warning": dd_at_warn,
                "exit_date": exit_date.date() if pd.notna(exit_date) else None,
                "lag_after_bottom_days": exit_lag,
                "max_probability_during_episode": float(prob.loc[onset:end].max()),
            })
            assert peak < onset <= bottom
            if pd.isna(first_warn):
                verdicts.append(f"{cname} episode {episode_number} / {mname}: no fresh warning transition.")
            else:
                timing = f"{abs(delay):.0f} days before onset" if delay < 0 else f"{delay:.0f} days after onset"
                verdicts.append(f"{cname} episode {episode_number} / {mname}: fresh warning {timing}; drawdown at warning {dd_at_warn:.1%}.")

taskB = pd.DataFrame(taskB_rows)
print(taskB.to_string(index=False))
print("\nEpisode-specific verdicts:")
for verdict in verdicts:
    print(" -", verdict)
save_csv(taskB, "taskB_event_study_episode_corrected", index=False)



TASK B — EPISODE-SPECIFIC EVENT STUDY
    crisis  episode          model episode_peak_date drawdown_onset_date bottom_date fresh_warning_date  lead_lag_vs_onset_days  drawdown_at_warning  exit_date  lag_after_bottom_days  max_probability_during_episode
  2008 GFC        1   HMM filtered        2007-10-09          2007-10-10  2008-11-20         2007-07-26                -76.0000              -0.0453 2009-06-09                    201                          1.0000
  2008 GFC        1 Logit risk-off        2007-10-09          2007-10-10  2008-11-20         2008-01-08                 90.0000              -0.1118 2009-04-03                    134                          1.0000
  2008 GFC        2   HMM filtered        2009-01-06          2009-01-07  2009-03-09               None                     NaN                  NaN 2009-06-09                     92                          1.0000
  2008 GFC        2 Logit risk-off        2009-01-06          2009-01-07  2009-03-09         2008-09-

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'/content/outputs/taskB_event_study_episode_corrected_20260922_052932_UTC.csv'

## TASK C — rule-based causal benchmarks

In [9]:
hr("TASK C — RULE-BASED BENCHMARKS")
# Causal: 200-day trailing MA uses only prices up to t.
rule_ma200 = (df["price"] < df["price"].rolling(200).mean()).astype(float)
rule_ma200[df["price"].rolling(200).mean().isna()] = np.nan
# Causal: drawdown from running peak is known at t.
rule_dd10 = (df["drawdown"] <= -0.10).astype(float)
rule_dd15 = (df["drawdown"] <= -0.15).astype(float)
# Causal: trailing 21d realized vol vs the EXPANDING 80th percentile of its own
# history up to t (no full-sample quantile → no look-ahead).
vol_thr = df["vol_21"].expanding(min_periods=252).quantile(0.80)
rule_vol = (df["vol_21"] > vol_thr).astype(float)
rule_vol[vol_thr.isna() | df["vol_21"].isna()] = np.nan

RULES = {"MA200 rule": rule_ma200, "Drawdown<=-10% rule": rule_dd10,
         "Drawdown<=-15% rule": rule_dd15, "Vol>expanding-80pct rule": rule_vol}

# Evaluate on the SAME dates as the models (where HMM filtered OOS exists).
common = df["hmm_filtered_riskoff"].notna()
rowsC = []
for rname, sig in RULES.items():
    m = common & sig.notna()
    res = evaluate(sig[m], df["risk_off"].astype(float)[m], rname)
    res.pop("ROC_AUC"); res.pop("PR_AUC"); res.pop("Brier"); res.pop("LogLoss")  # meaningless for hard 0/1 signals
    g_, n_, to = run_strategy((1 - sig), m)   # risk-off -> cash
    res.update({"CAGR_net": n_["CAGR"], "Sharpe_net": n_["Sharpe"], "MaxDD_net": n_["MaxDD"], "turnover": to})
    rowsC.append(res)
# model reference rows at 0.50 on the same dates
for mname, prob in NOWCASTS.items():
    m = common & prob.notna()
    res = evaluate(prob[m], df["risk_off"].astype(float)[m], f"{mname} (model, thr=0.50)")
    sig = (prob >= PROB_THRESHOLD).astype(float)
    g_, n_, to = run_strategy((1 - sig), m)
    res.update({"CAGR_net": n_["CAGR"], "Sharpe_net": n_["Sharpe"], "MaxDD_net": n_["MaxDD"], "turnover": to})
    rowsC.append(res)
taskC = pd.DataFrame(rowsC).set_index("model")
print(taskC.round(4).to_string())
save_csv(taskC.round(6), "taskC_rule_benchmarks")


TASK C — RULE-BASED BENCHMARKS


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


                                     n  base_rate  Precision  Recall     F1  FP_rate   TP    FP   FN    TN  CAGR_net  Sharpe_net  MaxDD_net  turnover  ROC_AUC  PR_AUC  Brier  LogLoss
model                                                                                                                                                                                 
MA200 rule                        8246     0.1277     0.4118  0.8044 0.5447   0.1682  847  1210  206  5983    0.0614      0.5719    -0.3095  226.0000      NaN     NaN    NaN      NaN
Drawdown<=-10% rule               8246     0.1277     0.2389  0.7189 0.3586   0.3353  757  2412  296  4781    0.0550      0.5440    -0.3596  116.0000      NaN     NaN    NaN      NaN
Drawdown<=-15% rule               8246     0.1277     0.2146  0.5147 0.3029   0.2758  542  1984  511  5209    0.0479      0.4402    -0.5040  112.0000      NaN     NaN    NaN      NaN
Vol>expanding-80pct rule          8246     0.1277     0.2964  0.6258 0.4023   0.2174 

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'/content/outputs/taskC_rule_benchmarks_20260922_052932_UTC.csv'

## TASK D — threshold sensitivity

In [10]:
hr("TASK D — THRESHOLD SENSITIVITY (strategy metrics net of costs)")
PROB_MODELS = {
    "hmm_filtered_riskoff": ("nowcast", df["risk_off"].astype(float)),
    "logit_riskoff_prob":   ("nowcast", df["risk_off"].astype(float)),
    "hmm_drawdown_prob":    ("forecast", df["target_drawdown"]),
    "logit_drawdown_prob":  ("forecast", df["target_drawdown"]),
    "gbm_drawdown_prob":    ("forecast", df["target_drawdown"]),
}
taskD_all = []
for col, (kind, target) in PROB_MODELS.items():
    prob = df[col]
    for thr in [0.50, 0.60, 0.70, 0.80]:
        m = prob.notna() & target.notna()
        y = target[m].astype(int).values
        yhat = (prob[m].values >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
        sig = (prob >= thr).astype(float)
        sig[prob.isna()] = np.nan
        g_, n_, to = run_strategy((1 - sig), prob.notna())
        taskD_all.append({
            "model": col, "target": kind, "threshold": thr,
            "precision": precision_score(y, yhat, zero_division=0),
            "recall": recall_score(y, yhat, zero_division=0),
            "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
            "mean_warning_duration_days": mean_run_length(sig),
            "max_drawdown": n_["MaxDD"], "cagr": n_["CAGR"],
            "sharpe": n_["Sharpe"], "turnover": to,
        })
taskD = pd.DataFrame(taskD_all)
print(taskD.round(4).to_string(index=False))
save_csv(taskD, "taskD_threshold_sensitivity", index=False)


TASK D — THRESHOLD SENSITIVITY (strategy metrics net of costs)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


               model   target  threshold  precision  recall    fpr  mean_warning_duration_days  max_drawdown   cagr  sharpe  turnover
hmm_filtered_riskoff  nowcast     0.5000     0.3417  0.7483 0.2110                     11.0865       -0.4799 0.0454  0.4640  416.0000
hmm_filtered_riskoff  nowcast     0.6000     0.3516  0.7085 0.1913                      9.6018       -0.5173 0.0464  0.4613  442.0000
hmm_filtered_riskoff  nowcast     0.7000     0.3612  0.6638 0.1718                      8.4130       -0.5217 0.0542  0.5118  460.0000
hmm_filtered_riskoff  nowcast     0.8000     0.3769  0.6002 0.1453                      6.4749       -0.6041 0.0535  0.4884  518.0000
  logit_riskoff_prob  nowcast     0.5000     0.5848  0.4501 0.0594                     13.8276       -0.3285 0.0598  0.4690  116.0000
  logit_riskoff_prob  nowcast     0.6000     0.5632  0.3551 0.0512                     10.7705       -0.4014 0.0570  0.4388  122.0000
  logit_riskoff_prob  nowcast     0.7000     0.5545  0.2879 0.

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'/content/outputs/taskD_threshold_sensitivity_20260922_052932_UTC.csv'

## TASK E — economic test as risk management, with transaction costs

In [11]:
hr(f"TASK E — STRATEGY COMPARISON (cost = {COST_BPS:.0f} bps per unit turnover)")
hmm_sig = (df["hmm_filtered_riskoff"] >= PROB_THRESHOLD).astype(float)
hmm_sig[df["hmm_filtered_riskoff"].isna()] = np.nan
maskE = df["hmm_filtered_riskoff"].notna() & rule_ma200.notna()   # same dates for all
STRATS = {
    "Buy & hold": pd.Series(1.0, index=df.index),
    "MA200 rule (cash when below)": (1 - rule_ma200),
    "HMM risk-managed, full cash": (1 - hmm_sig),
    "HMM risk-managed, 50% equity": (1 - 0.5 * hmm_sig),
}
rowsE = []
for sname, pos in STRATS.items():
    g_, n_, to = run_strategy(pos, maskE)
    rowsE.append({"strategy": sname,
                  "CAGR_gross": g_["CAGR"], "CAGR_net": n_["CAGR"],
                  "AnnVol_net": n_["AnnVol"], "Sharpe_gross": g_["Sharpe"],
                  "Sharpe_net": n_["Sharpe"], "MaxDD_gross": g_["MaxDD"],
                  "MaxDD_net": n_["MaxDD"], "turnover": to})
taskE = pd.DataFrame(rowsE).set_index("strategy")
print(taskE.round(4).to_string())
save_csv(taskE.round(6), "taskE_strategies_with_costs")

bh_mdd = taskE.loc["Buy & hold", "MaxDD_net"]
fc_mdd = taskE.loc["HMM risk-managed, full cash", "MaxDD_net"]
bh_cagr = taskE.loc["Buy & hold", "CAGR_net"]
fc_cagr = taskE.loc["HMM risk-managed, full cash", "CAGR_net"]
taskE_verdict = (f"Risk-management framing (computed): full-cash de-risking cuts max drawdown from "
                 f"{bh_mdd:.1%} (buy & hold) to {fc_mdd:.1%}, at a CAGR opportunity cost of "
                 f"{bh_cagr - fc_cagr:.2%}/yr net of {COST_BPS:.0f}bps costs "
                 f"({bh_cagr:.2%} vs {fc_cagr:.2%}).")
print("\n" + taskE_verdict)


TASK E — STRATEGY COMPARISON (cost = 10 bps per unit turnover)
                              CAGR_gross  CAGR_net  AnnVol_net  Sharpe_gross  Sharpe_net  MaxDD_gross  MaxDD_net  turnover
strategy                                                                                                                  
Buy & hold                        0.0899    0.0899      0.1852        0.5581      0.5581      -0.5678    -0.5678    0.0000
MA200 rule (cash when below)      0.0688    0.0614      0.1163        0.6315      0.5719      -0.2828    -0.3095  226.0000
HMM risk-managed, full cash       0.0588    0.0454      0.1085        0.5816      0.4640      -0.4257    -0.4799  416.0000
HMM risk-managed, 50% equity      0.0773    0.0705      0.1319        0.6313      0.5830      -0.4045    -0.4281  208.0000
[saved] /content/outputs/taskE_strategies_with_costs_20260922_052932_UTC.csv

Risk-management framing (computed): full-cash de-risking cuts max drawdown from -56.8% (buy & hold) to -48.0%, at a CAGR

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## SECTION F — LEAKAGE-CORRECTED SUPERVISED RESULTS (primary) + impact of the fix

In [12]:
hr("SECTION F — LEAKAGE-CORRECTED CLASSIFICATION")
rowsF = [
    evaluate(df["logit_riskoff_prob_purged"], df["risk_off"].astype(float), "Logit nowcast (purged)"),
    evaluate(df["logit_drawdown_prob_purged"], df["target_drawdown"], "Logit forecast (purged)"),
    evaluate(df["gbm_drawdown_prob_purged"], df["target_drawdown"], "GBM forecast (purged)"),
]
metricsF = pd.DataFrame([x for x in rowsF if x]).set_index("model")
print(metricsF.round(4).to_string())
save_csv(metricsF.round(6), "sectionF_classification_purged")

impact_rows = []
for label, leaky_col, purged_col, target in [
    ("Logit nowcast", "logit_riskoff_prob", "logit_riskoff_prob_purged", df["risk_off"].astype(float)),
    ("Logit forecast", "logit_drawdown_prob", "logit_drawdown_prob_purged", df["target_drawdown"]),
    ("GBM forecast", "gbm_drawdown_prob", "gbm_drawdown_prob_purged", df["target_drawdown"]),
]:
    shared = df[leaky_col].notna() & df[purged_col].notna() & target.notna()
    leaky_eval = evaluate(df.loc[shared, leaky_col], target[shared], "leaky")
    purged_eval = evaluate(df.loc[shared, purged_col], target[shared], "purged")
    assert leaky_eval["n"] == purged_eval["n"] == int(shared.sum())
    impact_rows.append({
        "model": label,
        "shared_n": int(shared.sum()),
        "shared_start": df.index[shared].min().date(),
        "shared_end": df.index[shared].max().date(),
        "AUC_leaky": leaky_eval["ROC_AUC"],
        "AUC_purged": purged_eval["ROC_AUC"],
        "AUC_delta": purged_eval["ROC_AUC"] - leaky_eval["ROC_AUC"],
        "Precision_leaky": leaky_eval["Precision"],
        "Precision_purged": purged_eval["Precision"],
        "Precision_delta": purged_eval["Precision"] - leaky_eval["Precision"],
    })

impactF = pd.DataFrame(impact_rows).set_index("model")
hr("SECTION F — LEAKAGE IMPACT ON IDENTICAL DATES")
print(impactF.round(4).to_string())
save_csv(impactF.round(6), "sectionF_leakage_impact_shared_dates")



SECTION F — LEAKAGE-CORRECTED CLASSIFICATION
                            n  base_rate  ROC_AUC  PR_AUC  Brier  LogLoss  Precision  Recall     F1  FP_rate   TP   FP    FN    TN
model                                                                                                                             
Logit nowcast (purged)   6419     0.1263   0.8619  0.4361 0.0976   0.3808     0.4863  0.3933 0.4349   0.0601  319  337   492  5271
Logit forecast (purged)  7154     0.1677   0.6308  0.3218 0.1496   0.5537     0.3461  0.1583 0.2173   0.0603  190  359  1010  5595
GBM forecast (purged)    7154     0.1677   0.5892  0.2123 0.1696   0.8363     0.2144  0.0967 0.1333   0.0714  116  425  1084  5529
[saved] /content/outputs/sectionF_classification_purged_20260922_052932_UTC.csv

SECTION F — LEAKAGE IMPACT ON IDENTICAL DATES
                shared_n shared_start  shared_end  AUC_leaky  AUC_purged  AUC_delta  Precision_leaky  Precision_purged  Precision_delta
model                               

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'/content/outputs/sectionF_leakage_impact_shared_dates_20260922_052932_UTC.csv'

## SECTION G — MULTIVARIATE FEATURES (VIX, credit spread, yield-curve slope)

In [13]:
# Rationale: the price-only models share their entire information set with a
# moving average, so they cannot out-inform it. This section adds series a price
# rule cannot see. Every exogenous series is reindexed to the S&P trading
# calendar, forward-filled (carry last published value), and LAGGED one trading
# day, so a feature dated t uses only information available through t-1. Momentum
# terms are diffs of the already-lagged series and are therefore also causal.
hr("SECTION G — MULTIVARIATE FEATURES")

def align_causal(s, lag=1):
    """Reindex to trading days, carry last known value forward, then lag.
    Result at t depends only on information available at/ before t-lag."""
    return s.reindex(df.index).ffill().shift(lag)

mv = pd.DataFrame(index=df.index)
mv_sources = {}

# VIX via yfinance (available from 1990). Known at close t -> lag 1 for prediction at t.
try:
    vraw = yf.download("^VIX", start=START_DATE, auto_adjust=False, progress=False)
    if vraw is not None and len(vraw):
        if isinstance(vraw.columns, pd.MultiIndex):
            vraw.columns = vraw.columns.get_level_values(0)
        vix = vraw["Close"].astype(float)
        mv["vix_level"] = align_causal(vix)
        mv["vix_chg_5"] = align_causal(vix).diff(5)     # diff of lagged series stays causal
        mv_sources["VIX (^VIX, yfinance)"] = f"{vix.index.min().date()} to {vix.index.max().date()}"
    else:
        print("[warn] ^VIX returned no data; VIX features skipped.")
except Exception as e:
    print(f"[warn] could not load ^VIX: {e}; VIX features skipped.")

# FRED series via pandas_datareader (no API key needed). If unavailable, skip and report.
def load_fred(series_id):
    try:
        from pandas_datareader import data as pdr
        s = pdr.DataReader(series_id, "fred", START_DATE)[series_id].astype(float)
        return s.dropna()
    except Exception as e:
        print(f"[warn] could not load FRED {series_id}: {e}; feature skipped. "
              f"(In Colab: pip install pandas_datareader)")
        return None

# High-yield credit spread (ICE BofA US HY OAS). Published next day -> lag 1. History from ~1996-12.
credit = load_fred("BAMLH0A0HYM2")
if credit is not None:
    mv["credit_level"]  = align_causal(credit)
    mv["credit_chg_21"] = align_causal(credit).diff(21)
    mv_sources["HY credit OAS (BAMLH0A0HYM2, FRED)"] = f"{credit.index.min().date()} to {credit.index.max().date()}"

# Yield-curve slope (10y minus 3m, percentage points). Daily -> lag 1.
curve = load_fred("T10Y3M")
if curve is not None:
    mv["curve_level"]  = align_causal(curve)
    mv["curve_chg_63"] = align_causal(curve).diff(63)
    mv_sources["Yield-curve slope (T10Y3M, FRED)"] = f"{curve.index.min().date()} to {curve.index.max().date()}"

# Keep only exogenous features with enough usable history for the
# expanding-window model. The FRED credit series in this run begins in 2023,
# so it is excluded rather than making every combined feature row unusable.
MIN_MV_HISTORY = MIN_TRAIN_DAYS
raw_mv_features = list(mv.columns)
MV_FEATURES = [
    col for col in raw_mv_features
    if mv[col].notna().sum() >= MIN_MV_HISTORY
]
skipped_mv_features = [
    col for col in raw_mv_features
    if col not in MV_FEATURES
]

if skipped_mv_features:
    print(
        f"Skipping short-history features (< {MIN_MV_HISTORY} usable rows): "
        f"{skipped_mv_features}"
    )

df[MV_FEATURES] = mv[MV_FEATURES]

print("Exogenous sources loaded:")
for k, v in mv_sources.items():
    print(f"  - {k}: {v}")
print(f"Multivariate features used: {MV_FEATURES if MV_FEATURES else 'NONE'}")

if MV_FEATURES:
    # Full feature set = price features + sufficiently long exogenous features.
    ALL_FEATURES = FEATURES + MV_FEATURES
    Xmv = df[ALL_FEATURES]

    df["logit_riskoff_prob_mv"] = walk_forward_predict(
        Xmv, df["risk_off"].astype(float), make_logit,
        MIN_TRAIN_DAYS, REFIT_EVERY, known_pos=RISKOFF_KNOWN_POS
    )
    df["gbm_riskoff_prob_mv"] = walk_forward_predict(
        Xmv, df["risk_off"].astype(float), make_gbm,
        MIN_TRAIN_DAYS, REFIT_EVERY, known_pos=RISKOFF_KNOWN_POS
    )
    df["logit_drawdown_prob_mv"] = walk_forward_predict(
        Xmv, df["target_drawdown"], make_logit,
        MIN_TRAIN_DAYS, REFIT_EVERY, known_pos=FWD_KNOWN_POS
    )
    df["gbm_drawdown_prob_mv"] = walk_forward_predict(
        Xmv, df["target_drawdown"], make_gbm,
        MIN_TRAIN_DAYS, REFIT_EVERY, known_pos=FWD_KNOWN_POS
    )

    for c in [
        "logit_riskoff_prob_mv",
        "gbm_riskoff_prob_mv",
        "logit_drawdown_prob_mv",
        "gbm_drawdown_prob_mv",
    ]:
        assert df[c].dropna().between(0, 1).all(), c

    rowsG = [
        evaluate(df["logit_riskoff_prob_mv"], df["risk_off"].astype(float),
                 "Logit nowcast (multivariate)"),
        evaluate(df["gbm_riskoff_prob_mv"], df["risk_off"].astype(float),
                 "GBM nowcast (multivariate)"),
        evaluate(df["logit_drawdown_prob_mv"], df["target_drawdown"],
                 "Logit forecast (multivariate)"),
        evaluate(df["gbm_drawdown_prob_mv"], df["target_drawdown"],
                 "GBM forecast (multivariate)"),
    ]
    valid_rowsG = [row for row in rowsG if row is not None]

    if not valid_rowsG:
        mv_verdict = (
            "Multivariate features loaded, but no valid out-of-sample "
            "predictions were produced. Section G was skipped."
        )
        print(mv_verdict)
    else:
        hr("SECTION G — MULTIVARIATE CLASSIFICATION")
        metricsG = pd.DataFrame(valid_rowsG).set_index("model")
        print(metricsG.round(4).to_string())
        save_csv(metricsG.round(6), "sectionG_classification_multivariate")

        common_mv = (
            df["logit_riskoff_prob_mv"].notna()
            & df["logit_riskoff_prob_purged"].notna()
            & rule_ma200.notna()
        )

        if common_mv.any():
            contenders = {
                "MA200 rule": 1 - rule_ma200,
                "Logit nowcast price-only (purged)": 1 - (
                    df["logit_riskoff_prob_purged"] >= PROB_THRESHOLD
                ).astype(float),
                "Logit nowcast multivariate": 1 - (
                    df["logit_riskoff_prob_mv"] >= PROB_THRESHOLD
                ).astype(float),
                "GBM nowcast multivariate": 1 - (
                    df["gbm_riskoff_prob_mv"] >= PROB_THRESHOLD
                ).astype(float),
            }

            probs_for_auc = {
                "MA200 rule": None,
                "Logit nowcast price-only (purged)": df["logit_riskoff_prob_purged"],
                "Logit nowcast multivariate": df["logit_riskoff_prob_mv"],
                "GBM nowcast multivariate": df["gbm_riskoff_prob_mv"],
            }

            rowsH = []
            for nm, pos in contenders.items():
                g_, n_, to = run_strategy(pos, common_mv)
                rec = {
                    "model": nm,
                    "CAGR_net": n_["CAGR"],
                    "Sharpe_net": n_["Sharpe"],
                    "MaxDD_net": n_["MaxDD"],
                    "turnover": to,
                }
                pr = probs_for_auc[nm]
                if pr is not None:
                    e = evaluate(
                        pr[common_mv],
                        df["risk_off"].astype(float)[common_mv],
                        nm,
                    )
                    rec.update({
                        "ROC_AUC": e["ROC_AUC"],
                        "Precision": e["Precision"],
                        "Recall": e["Recall"],
                    })
                rowsH.append(rec)

            _, n_, to = run_strategy(pd.Series(1.0, index=df.index), common_mv)
            rowsH.append({
                "model": "Buy & hold",
                "CAGR_net": n_["CAGR"],
                "Sharpe_net": n_["Sharpe"],
                "MaxDD_net": n_["MaxDD"],
                "turnover": to,
            })

            headG = pd.DataFrame(rowsH).set_index("model")
            print(headG.round(4).to_string())
            save_csv(headG.round(6), "sectionG_head_to_head")

        mv_verdict = (
            "Multivariate models use VIX and yield-curve features. "
            "The short-history credit-spread features were excluded."
        )
else:
    mv_verdict = "No exogenous feature has enough history; multivariate section skipped."

print("\n" + mv_verdict)


SECTION G — MULTIVARIATE FEATURES


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Skipping short-history features (< 1000 usable rows): ['credit_level', 'credit_chg_21']
Exogenous sources loaded:
  - VIX (^VIX, yfinance): 1990-01-02 to 2026-09-21
  - HY credit OAS (BAMLH0A0HYM2, FRED): 2023-09-22 to 2026-09-18
  - Yield-curve slope (T10Y3M, FRED): 1990-01-02 to 2026-09-21
Multivariate features used: ['vix_level', 'vix_chg_5', 'curve_level', 'curve_chg_63']


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



SECTION G — MULTIVARIATE CLASSIFICATION
                                  n  base_rate  ROC_AUC  PR_AUC  Brier  LogLoss  Precision  Recall     F1  FP_rate   TP   FP    FN    TN
model                                                                                                                                   
Logit nowcast (multivariate)   6419     0.1263   0.6767  0.3241 0.1273   0.5548     0.3696  0.3687 0.3691   0.0909  299  510   512  5098
GBM nowcast (multivariate)     6419     0.1263   0.7906  0.3311 0.1271   0.6037     0.3677  0.3477 0.3574   0.0865  282  485   529  5123
Logit forecast (multivariate)  7154     0.1677   0.5282  0.2504 0.1690   0.6793     0.3310  0.1600 0.2157   0.0652  192  388  1008  5566
GBM forecast (multivariate)    7154     0.1677   0.5365  0.1878 0.1735   0.9272     0.1948  0.0683 0.1012   0.0569   82  339  1118  5615
[saved] /content/outputs/sectionG_classification_multivariate_20260922_052932_UTC.csv
                                   CAGR_net  Sharpe

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section H — data provenance and investable strategy returns

Classification continues to use the S&P 500 index. Economic tests use adjusted SPY prices as a tradable total-return proxy, next-day execution, a lagged 13-week Treasury yield as the cash return, and explicit turnover costs. Yahoo and ordinary FRED downloads are latest-vintage sources, not archival point-in-time databases; the provenance table records that limitation.

In [14]:
hr("SECTION H — DATA PROVENANCE")
provenance = pd.DataFrame([
    {"dataset": "S&P 500 index", "identifier": "^GSPC / Yahoo Finance", "use": "labels and price features", "point_in_time_archive": False, "known_issue": "latest vendor history; not an exchange-grade frozen snapshot"},
    {"dataset": "SPY adjusted close", "identifier": "SPY / Yahoo Finance", "use": "investable total-return proxy", "point_in_time_archive": False, "known_issue": "vendor-adjusted history; verify against CRSP for publication"},
    {"dataset": "VIX", "identifier": "^VIX / Yahoo Finance", "use": "lagged feature", "point_in_time_archive": False, "known_issue": "latest vendor history"},
    {"dataset": "FRED macro series", "identifier": "BAMLH0A0HYM2, T10Y3M", "use": "lagged features", "point_in_time_archive": False, "known_issue": "latest vintage; use ALFRED vintages for a confirmatory paper"},
    {"dataset": "13-week Treasury yield", "identifier": "^IRX / Yahoo Finance", "use": "lagged cash return", "point_in_time_archive": False, "known_issue": "approximation; replace with a documented total-return T-bill series"},
])
print(provenance.to_string(index=False))
save_csv(provenance, "data_provenance", index=False)

def close_series(downloaded):
    if isinstance(downloaded.columns, pd.MultiIndex):
        downloaded = downloaded.copy()
        downloaded.columns = downloaded.columns.get_level_values(0)
    return downloaded["Close"].astype(float)

spy_raw = yf.download("SPY", start="1993-01-01", auto_adjust=True, progress=False)
if spy_raw is None or len(spy_raw) == 0:
    raise RuntimeError("SPY download failed; publication strategy test cannot use index price returns as a silent substitute.")
spy_price = close_series(spy_raw).reindex(df.index).ffill()
spy_ret = spy_price.pct_change()

irx_raw = yf.download("^IRX", start="1993-01-01", auto_adjust=False, progress=False)
if irx_raw is not None and len(irx_raw):
    irx_yield = close_series(irx_raw).reindex(df.index).ffill().shift(1)
    cash_ret = (irx_yield / 100.0 / 252.0).fillna(0.0)
    CASH_SOURCE = "lagged ^IRX / 252"
else:
    cash_ret = pd.Series(0.0, index=df.index)
    CASH_SOURCE = "zero cash return because ^IRX was unavailable"

def investable_returns(position, cost_bps):
    '''Close-t signal, position effective on t+1; uninvested capital earns cash.'''
    pos = position.reindex(df.index).shift(1).clip(0, 1)
    turnover = pos.diff().abs().fillna(0.0)
    gross = pos * spy_ret + (1.0 - pos) * cash_ret
    net = gross - (cost_bps / 1e4) * turnover
    return net, turnover

publication_positions = {
    "Buy & hold": pd.Series(1.0, index=df.index),
    "MA200": 1.0 - rule_ma200,
    "Price-only Logit (purged)": 1.0 - (df["logit_riskoff_prob_purged"] >= PROB_THRESHOLD).astype(float),
    "HMM filtered": 1.0 - (df["hmm_filtered_riskoff"] >= PROB_THRESHOLD).astype(float),
}
if "logit_riskoff_prob_mv" in df:
    publication_positions["Multivariate Logit"] = 1.0 - (df["logit_riskoff_prob_mv"] >= PROB_THRESHOLD).astype(float)

shared_strategy_mask = spy_ret.notna()
for position in publication_positions.values():
    shared_strategy_mask &= position.notna()
shared_strategy_mask &= df.index >= RETROSPECTIVE_TEST_START

strategy_daily_returns = {}
strategy_rows = []
for name, position in publication_positions.items():
    net, turnover = investable_returns(position, COST_BPS)
    net = net[shared_strategy_mask].dropna()
    strategy_daily_returns[name] = net
    _, stats = strategy_stats(net)
    strategy_rows.append({"strategy": name, **stats, "turnover": float(turnover[shared_strategy_mask].sum())})

publication_strategy = pd.DataFrame(strategy_rows).set_index("strategy")
print(f"Cash return source: {CASH_SOURCE}")
print(f"Common retrospective span: {df.index[shared_strategy_mask].min().date()} to {df.index[shared_strategy_mask].max().date()}")
print(publication_strategy.round(4).to_string())
save_csv(publication_strategy.round(6), "publication_investable_strategy")


SECTION H — DATA PROVENANCE
               dataset            identifier                           use  point_in_time_archive                                                         known_issue
         S&P 500 index ^GSPC / Yahoo Finance     labels and price features                  False        latest vendor history; not an exchange-grade frozen snapshot
    SPY adjusted close   SPY / Yahoo Finance investable total-return proxy                  False        vendor-adjusted history; verify against CRSP for publication
                   VIX  ^VIX / Yahoo Finance                lagged feature                  False                                               latest vendor history
     FRED macro series  BAMLH0A0HYM2, T10Y3M               lagged features                  False        latest vintage; use ALFRED vintages for a confirmatory paper
13-week Treasury yield  ^IRX / Yahoo Finance            lagged cash return                  False approximation; replace with a documented to

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Cash return source: lagged ^IRX / 252
Common retrospective span: 2016-01-04 to 2026-09-21
                            CAGR  AnnVol  Sharpe   MaxDD  turnover
strategy                                                          
Buy & hold                0.1510  0.1773  0.8838 -0.3372    0.0000
MA200                     0.1035  0.1168  0.9040 -0.1968   62.0000
Price-only Logit (purged) 0.1251  0.1577  0.8283 -0.2674   24.0000
HMM filtered              0.0916  0.1054  0.8871 -0.1700  118.0000
Multivariate Logit        0.0938  0.1445  0.6942 -0.3458   78.0000
[saved] /content/outputs/publication_investable_strategy_20260922_052932_UTC.csv


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/datetimes.py:666: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x7e2c695206d0>
  converted = ints_to_pydatetime(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'/content/outputs/publication_investable_strategy_20260922_052932_UTC.csv'

## Section I — same-date temporal validation

All contenders in each table are restricted to one joint mask. The 2016–end segment is a retrospective temporal slice, not an untouched test, because its outcomes were viewed during model development.

In [15]:
hr("SECTION I — SAME-DATE RETROSPECTIVE TEMPORAL VALIDATION")

nowcast_probs = {
    "MA200 hard signal": rule_ma200,
    "HMM filtered": df["hmm_filtered_riskoff"],
    "Price-only Logit (purged)": df["logit_riskoff_prob_purged"],
}
if "logit_riskoff_prob_mv" in df:
    nowcast_probs["Multivariate Logit"] = df["logit_riskoff_prob_mv"]
if "gbm_riskoff_prob_mv" in df:
    nowcast_probs["Multivariate GBM"] = df["gbm_riskoff_prob_mv"]

joint_nowcast = df["risk_off"].notna()
for prob in nowcast_probs.values():
    joint_nowcast &= prob.notna()

temporal_rows = []
for period, (start, end) in SUBPERIODS.items():
    period_mask = joint_nowcast & (df.index >= start)
    if end is not None:
        period_mask &= df.index <= end
    if period_mask.sum() == 0 or df.loc[period_mask, "risk_off"].nunique() < 2:
        continue
    for name, prob in nowcast_probs.items():
        result = evaluate(prob[period_mask], df.loc[period_mask, "risk_off"].astype(float), name)
        temporal_rows.append({"period": period, **result})

temporal_nowcast = pd.DataFrame(temporal_rows).set_index(["period", "model"])
print(temporal_nowcast.round(4).to_string())
save_csv(temporal_nowcast.round(6), "temporal_nowcast_same_dates")

forecast_probs = {
    "Logit forecast (purged)": df["logit_drawdown_prob_purged"],
    "GBM forecast (purged)": df["gbm_drawdown_prob_purged"],
    "HMM simulation": df["hmm_drawdown_prob"],
}
if "logit_drawdown_prob_mv" in df:
    forecast_probs["Multivariate Logit forecast"] = df["logit_drawdown_prob_mv"]
if "gbm_drawdown_prob_mv" in df:
    forecast_probs["Multivariate GBM forecast"] = df["gbm_drawdown_prob_mv"]

joint_forecast = df["target_drawdown"].notna()
for prob in forecast_probs.values():
    joint_forecast &= prob.notna()
joint_forecast &= df.index >= RETROSPECTIVE_TEST_START

forecast_rows = []
for name, prob in forecast_probs.items():
    result = evaluate(prob[joint_forecast], df.loc[joint_forecast, "target_drawdown"], name)
    forecast_rows.append(result)
retrospective_forecast = pd.DataFrame(forecast_rows).set_index("model")
print("\nRetrospective 2016–end forecast comparison on identical dates:")
print(retrospective_forecast.round(4).to_string())
save_csv(retrospective_forecast.round(6), "retrospective_forecast_same_dates")


SECTION I — SAME-DATE RETROSPECTIVE TEMPORAL VALIDATION
                                                     n  base_rate  ROC_AUC  PR_AUC  Brier  LogLoss  Precision  Recall     F1  FP_rate   TP   FP   FN    TN
period                 model                                                                                                                              
early_oos_1994_2007    MA200 hard signal          1710     0.1901   0.7972  0.3974 0.2427   8.7474     0.4308  0.8615 0.5744   0.2671  280  370   45  1015
                       HMM filtered               1710     0.1901   0.8110  0.3991 0.2002   0.8359     0.4187  0.7292 0.5320   0.2375  237  329   88  1056
                       Price-only Logit (purged)  1710     0.1901   0.8036  0.3872 0.1745   0.6510     0.4160  0.5108 0.4586   0.1682  166  233  159  1152
                       Multivariate Logit         1710     0.1901   0.4886  0.2867 0.2347   1.1559     0.2958  0.3477 0.3197   0.1942  113  269  212  1116
             

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'/content/outputs/retrospective_forecast_same_dates_20260922_052932_UTC.csv'

## Section J — block-bootstrap uncertainty

Daily labels, forecasts, and returns are serially dependent. The stationary bootstrap below resamples blocks with mean length 63 trading days and reports percentile intervals. It also estimates paired AUC differences against MA200 on identical dates.

In [16]:
def stationary_bootstrap_indices(n, mean_block, rng):
    p_new = 1.0 / mean_block
    indices = np.empty(n, dtype=int)
    current = int(rng.integers(0, n))
    for i in range(n):
        if i == 0 or rng.random() < p_new:
            current = int(rng.integers(0, n))
        else:
            current = (current + 1) % n
        indices[i] = current
    return indices

def percentile_interval(values, level=0.95):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    alpha = 1.0 - level
    return tuple(np.quantile(values, [alpha / 2, 1 - alpha / 2]))

rng = np.random.default_rng(20260714)
test_mask = joint_nowcast & (df.index >= RETROSPECTIVE_TEST_START)
y_test = df.loc[test_mask, "risk_off"].astype(int).to_numpy()
prob_arrays = {name: prob[test_mask].to_numpy() for name, prob in nowcast_probs.items()}

bootstrap_auc = {name: [] for name in prob_arrays}
paired_auc_delta_vs_ma200 = {name: [] for name in prob_arrays if name != "MA200 hard signal"}
for _ in range(BOOTSTRAP_REPS):
    ix = stationary_bootstrap_indices(len(y_test), BOOTSTRAP_MEAN_BLOCK, rng)
    yb = y_test[ix]
    if np.unique(yb).size < 2:
        continue
    auc_ma = roc_auc_score(yb, prob_arrays["MA200 hard signal"][ix])
    for name, values in prob_arrays.items():
        auc = roc_auc_score(yb, values[ix])
        bootstrap_auc[name].append(auc)
        if name != "MA200 hard signal":
            paired_auc_delta_vs_ma200[name].append(auc - auc_ma)

uncertainty_rows = []
for name, values in prob_arrays.items():
    observed = roc_auc_score(y_test, values)
    lo, hi = percentile_interval(bootstrap_auc[name])
    row = {"model": name, "observed_auc": observed, "auc_ci_low": lo, "auc_ci_high": hi}
    if name != "MA200 hard signal":
        dlo, dhi = percentile_interval(paired_auc_delta_vs_ma200[name])
        row.update({"auc_delta_vs_ma200": observed - roc_auc_score(y_test, prob_arrays["MA200 hard signal"]),
                    "delta_ci_low": dlo, "delta_ci_high": dhi})
    uncertainty_rows.append(row)

auc_uncertainty = pd.DataFrame(uncertainty_rows).set_index("model")
print(auc_uncertainty.round(4).to_string())
save_csv(auc_uncertainty.round(6), "bootstrap_auc_uncertainty")

def path_metrics(returns):
    returns = np.asarray(returns, dtype=float)
    wealth = np.cumprod(1.0 + returns)
    cagr = wealth[-1] ** (252.0 / len(returns)) - 1.0
    vol = np.std(returns, ddof=1) * np.sqrt(252)
    sharpe = np.mean(returns) * 252 / vol if vol > 0 else np.nan
    maxdd = np.min(wealth / np.maximum.accumulate(wealth) - 1.0)
    return cagr, sharpe, maxdd

ma_returns = strategy_daily_returns["MA200"].to_numpy()
model_returns = strategy_daily_returns["Price-only Logit (purged)"].to_numpy()
assert len(ma_returns) == len(model_returns)
metric_deltas = {"CAGR": [], "Sharpe": [], "MaxDD": []}
for _ in range(BOOTSTRAP_REPS):
    ix = stationary_bootstrap_indices(len(ma_returns), BOOTSTRAP_MEAN_BLOCK, rng)
    ma_stats = path_metrics(ma_returns[ix])
    model_stats = path_metrics(model_returns[ix])
    for key, model_value, ma_value in zip(metric_deltas, model_stats, ma_stats):
        metric_deltas[key].append(model_value - ma_value)

strategy_uncertainty_rows = []
observed_delta = np.array(path_metrics(model_returns)) - np.array(path_metrics(ma_returns))
for key, observed in zip(metric_deltas, observed_delta):
    lo, hi = percentile_interval(metric_deltas[key])
    strategy_uncertainty_rows.append({"metric": key, "model_minus_ma200": observed, "ci_low": lo, "ci_high": hi})
strategy_uncertainty = pd.DataFrame(strategy_uncertainty_rows).set_index("metric")
print("\nPaired strategy differences, price-only Logit minus MA200:")
print(strategy_uncertainty.round(4).to_string())
save_csv(strategy_uncertainty.round(6), "bootstrap_strategy_difference")

                           observed_auc  auc_ci_low  auc_ci_high  auc_delta_vs_ma200  delta_ci_low  delta_ci_high
model                                                                                                            
MA200 hard signal                0.8442      0.6969       0.9442                 NaN           NaN            NaN
HMM filtered                     0.8814      0.8209       0.9423              0.0372       -0.0505         0.1672
Price-only Logit (purged)        0.8229      0.6394       0.9553             -0.0213       -0.0839         0.0345
Multivariate Logit               0.5975      0.3850       0.8475             -0.2467       -0.3864        -0.0037
Multivariate GBM                 0.8076      0.5959       0.9300             -0.0366       -0.1227         0.0226
[saved] /content/outputs/bootstrap_auc_uncertainty_20260922_052932_UTC.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Paired strategy differences, price-only Logit minus MA200:
        model_minus_ma200  ci_low  ci_high
metric                                    
CAGR               0.0216 -0.0250   0.0714
Sharpe            -0.0757 -0.4134   0.2619
MaxDD             -0.0706 -0.2005   0.0241
[saved] /content/outputs/bootstrap_strategy_difference_20260922_052932_UTC.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'/content/outputs/bootstrap_strategy_difference_20260922_052932_UTC.csv'

## Section K — calibration, cost sensitivity, and placebo tests

The calibration table checks whether predicted probabilities match observed frequencies. Strategy costs are varied from 0 to 50 bps. Circular-shift placebo tests preserve much of the target's serial structure while breaking its alignment with predictions; Holm adjustment controls family-wise error across the tested nowcasts.

In [17]:
hr("SECTION K — CALIBRATION")
calibration_models = {
    name: prob for name, prob in nowcast_probs.items()
    if "hard signal" not in name
}
calibration_rows = []
fig, ax = plt.subplots(figsize=(7, 6))
for name, prob in calibration_models.items():
    mask = prob.notna() & df["risk_off"].notna() & (df.index >= RETROSPECTIVE_TEST_START)
    tmp = pd.DataFrame({"prob": prob[mask], "y": df.loc[mask, "risk_off"].astype(int)})
    tmp["bin"] = pd.qcut(tmp["prob"], q=10, duplicates="drop")
    grouped = tmp.groupby("bin", observed=True).agg(mean_probability=("prob", "mean"), observed_rate=("y", "mean"), n=("y", "size"))
    ece = np.average(np.abs(grouped["mean_probability"] - grouped["observed_rate"]), weights=grouped["n"])
    calibration_rows.append({"model": name, "n": len(tmp), "Brier": brier_score_loss(tmp["y"], tmp["prob"]), "ECE": ece})
    ax.plot(grouped["mean_probability"], grouped["observed_rate"], marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="perfect calibration")
ax.set(xlabel="Mean predicted probability", ylabel="Observed risk-off rate", title="Retrospective calibration, 2016–end")
ax.legend(fontsize=8)
fig.tight_layout()
calibration_path = os.path.join(OUT, f"calibration_{STAMP}.png")
fig.savefig(calibration_path, dpi=180)
plt.close(fig)
SAVED_FILES.append(calibration_path)
calibration_summary = pd.DataFrame(calibration_rows).set_index("model")
print(calibration_summary.round(4).to_string())
save_csv(calibration_summary.round(6), "calibration_summary")

hr("SECTION K — COST SENSITIVITY")
cost_rows = []
for cost in COST_GRID_BPS:
    for name, position in publication_positions.items():
        net, turnover = investable_returns(position, cost)
        net = net[shared_strategy_mask].dropna()
        _, stats = strategy_stats(net)
        cost_rows.append({"cost_bps": cost, "strategy": name, **stats,
                          "turnover": float(turnover[shared_strategy_mask].sum())})
cost_sensitivity = pd.DataFrame(cost_rows).set_index(["cost_bps", "strategy"])
print(cost_sensitivity.round(4).to_string())
save_csv(cost_sensitivity.round(6), "strategy_cost_sensitivity")

def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    order = np.argsort(pvalues)
    adjusted_sorted = np.maximum.accumulate((len(pvalues) - np.arange(len(pvalues))) * pvalues[order])
    adjusted = np.empty_like(adjusted_sorted)
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)
    return adjusted

rng_placebo = np.random.default_rng(314159)
placebo_rows = []
n_test = len(y_test)
valid_shifts = np.arange(BOOTSTRAP_MEAN_BLOCK, n_test - BOOTSTRAP_MEAN_BLOCK)
for name, values in prob_arrays.items():
    observed = roc_auc_score(y_test, values)
    null_auc = []
    for shift in rng_placebo.choice(valid_shifts, size=PLACEBO_REPS, replace=True):
        null_auc.append(roc_auc_score(np.roll(y_test, int(shift)), values))
    p_raw = (1 + np.sum(np.asarray(null_auc) >= observed)) / (PLACEBO_REPS + 1)
    placebo_rows.append({"model": name, "observed_auc": observed, "null_auc_mean": np.mean(null_auc), "p_raw": p_raw})
placebo_tests = pd.DataFrame(placebo_rows)
placebo_tests["p_holm"] = holm_adjust(placebo_tests["p_raw"].to_numpy())
placebo_tests = placebo_tests.set_index("model")
print("\nCircular-shift placebo tests:")
print(placebo_tests.round(4).to_string())
save_csv(placebo_tests.round(6), "placebo_tests_holm")


SECTION K — CALIBRATION


/usr/local/lib/python3.13/dist-packages/matplotlib/_api/__init__.py:126: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x7e2c696b16c0>
  if val not in values:
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


                              n  Brier    ECE
model                                        
HMM filtered               2694 0.1516 0.1679
Price-only Logit (purged)  2694 0.0607 0.0137
Multivariate Logit         2694 0.1010 0.1226
Multivariate GBM           2694 0.1169 0.0932
[saved] /content/outputs/calibration_summary_20260922_052932_UTC.csv

SECTION K — COST SENSITIVITY
                                     CAGR  AnnVol  Sharpe   MaxDD  turnover
cost_bps strategy                                                          
0        Buy & hold                0.1510  0.1773  0.8838 -0.3372    0.0000
         MA200                     0.1099  0.1167  0.9541 -0.1895   62.0000
         Price-only Logit (purged) 0.1276  0.1578  0.8422 -0.2667   24.0000
         HMM filtered              0.1037  0.1053  0.9924 -0.1555  118.0000
         Multivariate Logit        0.1018  0.1444  0.7452 -0.3266   78.0000
5        Buy & hold                0.1510  0.1773  0.8838 -0.3372    0.0000
         MA200   

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


Circular-shift placebo tests:
                           observed_auc  null_auc_mean  p_raw  p_holm
model                                                                
MA200 hard signal                0.8442         0.4805 0.0080  0.0320
HMM filtered                     0.8814         0.4809 0.0030  0.0150
Price-only Logit (purged)        0.8229         0.4860 0.0220  0.0659
Multivariate Logit               0.5975         0.4984 0.3946  0.3946
Multivariate GBM                 0.8076         0.4920 0.0579  0.1159
[saved] /content/outputs/placebo_tests_holm_20260922_052932_UTC.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'/content/outputs/placebo_tests_holm_20260922_052932_UTC.csv'

## Section L — external-market validation

This is a pre-specified transport test of the fixed price-only logistic specification. Each market receives its own causally dated 20% regime labels and expanding-window model. It is classification evidence only: index price returns are not used to claim investable performance.

In [18]:
def build_market_frame(ticker):
    raw_market = yf.download(ticker, start=START_DATE, auto_adjust=True, progress=False)
    if raw_market is None or len(raw_market) == 0:
        return None
    px = close_series(raw_market).dropna()
    market = pd.DataFrame(index=px.index)
    market["price"] = px
    market["log_ret"] = np.log(px).diff()
    market = market.dropna().copy()
    if len(market) <= MIN_TRAIN_DAYS + 252:
        return None
    regime, confirm = regime_dating_with_confirm(market["price"], BEAR_RULE_THRESHOLD)
    market["risk_off"] = 1 - regime
    r = market["log_ret"]
    market["ret_21"] = r.rolling(21).sum()
    market["ret_63"] = r.rolling(63).sum()
    market["ret_126"] = r.rolling(126).sum()
    market["ret_252"] = r.rolling(252).sum()
    market["vol_21"] = r.rolling(21).std() * np.sqrt(252)
    market["vol_63"] = r.rolling(63).std() * np.sqrt(252)
    market["mom_50_200"] = market["price"].rolling(50).mean() / market["price"].rolling(200).mean() - 1
    market["dd_from_252high"] = market["price"] / market["price"].rolling(252).max() - 1
    market["ma200_signal"] = (market["price"] < market["price"].rolling(200).mean()).astype(float)
    market.loc[market["price"].rolling(200).mean().isna(), "ma200_signal"] = np.nan
    market["logit_prob"] = walk_forward_predict(
        market[FEATURES], market["risk_off"].astype(float), make_logit,
        MIN_TRAIN_DAYS, REFIT_EVERY, known_pos=confirm,
    )
    return market

external_rows = []
external_failures = []
for market_name, ticker in EXTERNAL_MARKETS.items():
    try:
        market = build_market_frame(ticker)
        if market is None:
            external_failures.append({"market": market_name, "ticker": ticker, "reason": "download failed or insufficient history"})
            continue
        shared = market[["risk_off", "ma200_signal", "logit_prob"]].notna().all(axis=1)
        shared &= market.index >= RETROSPECTIVE_TEST_START
        if shared.sum() == 0 or market.loc[shared, "risk_off"].nunique() < 2:
            external_failures.append({"market": market_name, "ticker": ticker, "reason": "no evaluable common retrospective dates"})
            continue
        y = market.loc[shared, "risk_off"].astype(int)
        for model_name, probability in {
            "MA200 hard signal": market.loc[shared, "ma200_signal"],
            "Price-only Logit (purged)": market.loc[shared, "logit_prob"],
        }.items():
            result = evaluate(probability, y, model_name)
            external_rows.append({"market": market_name, "ticker": ticker,
                                  "start": market.index[shared].min().date(),
                                  "end": market.index[shared].max().date(), **result})
    except Exception as exc:
        external_failures.append({"market": market_name, "ticker": ticker, "reason": repr(exc)})

external_validation = pd.DataFrame(external_rows)
if len(external_validation):
    external_validation = external_validation.set_index(["market", "model"])
    print(external_validation.round(4).to_string())
    save_csv(external_validation.round(6), "external_market_validation")
if external_failures:
    external_failure_table = pd.DataFrame(external_failures)
    print("\nExternal-market failures:")
    print(external_failure_table.to_string(index=False))
    save_csv(external_failure_table, "external_market_failures", index=False)

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/datetimes.py:666: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x7e2c6945a200>
  converted = ints_to_pydatetime(
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/datetimes.py:666: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x7e2c681a5990>
  converted = ints_to_pydatetime(
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/datetimes.py:666: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x7e2c681a5a80>
  converted = ints_to_pydatetime(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/datetimes.py:666: ResourceWarning: 

                                      ticker       start         end     n  base_rate  ROC_AUC  PR_AUC  Brier  LogLoss  Precision  Recall     F1  FP_rate   TP   FP   FN    TN
market     model                                                                                                                                                              
FTSE 100   MA200 hard signal           ^FTSE  2016-01-04  2026-09-21  2707     0.1829   0.6085  0.2353 0.3003  10.8251     0.2956  0.4646 0.3614   0.2477  230  548  265  1664
           Price-only Logit (purged)   ^FTSE  2016-01-04  2026-09-21  2707     0.1829   0.6105  0.2361 0.1782   0.5585     0.2479  0.1192 0.1610   0.0809   59  179  436  2033
DAX        MA200 hard signal          ^GDAXI  2016-01-04  2026-09-21  2721     0.1735   0.8615  0.5003 0.1474   5.3118     0.5470  0.8750 0.6732   0.1521  413  342   59  1907
           Price-only Logit (purged)  ^GDAXI  2016-01-04  2026-09-21  2721     0.1735   0.7673  0.4577 0.1187   0.3899     0.

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section M — trial registry and publication gate

The gate distinguishes checks the notebook can perform from evidence that must come from future data or an independent researcher. A failed gate is a result, not an error.

In [19]:
KNOWN_TRIAL_FAMILIES = [
    "HMM nowcast", "price-only Logit nowcast", "HMM simulation forecast",
    "price-only Logit forecast", "price-only GBM forecast",
    "MA200", "drawdown 10%", "drawdown 15%", "volatility 80th percentile",
    "thresholds 0.50/0.60/0.70/0.80", "full-cash/half-equity sizing",
    "multivariate Logit/GBM nowcast and forecast",
]
trial_registry = pd.DataFrame({"known_trial_family": KNOWN_TRIAL_FAMILIES})
save_csv(trial_registry, "known_trial_registry", index=False)

external_markets_completed = 0
if "external_validation" in globals() and len(external_validation):
    external_markets_completed = external_validation.reset_index()["market"].nunique()

publication_gate = pd.DataFrame([
    {"gate": "Notebook runs from clean kernel source", "passed": True, "evidence_or_action": "Section G loader restored; no stale output state"},
    {"gate": "Causal purged training labels", "passed": True, "evidence_or_action": "known_pos purge for regime and 63-day labels"},
    {"gate": "Leaky/purged comparison uses identical dates", "passed": True, "evidence_or_action": "Section F joint masks and equal-n assertion"},
    {"gate": "Episode-specific event warnings", "passed": True, "evidence_or_action": "fresh off-to-on transitions required"},
    {"gate": "Investable total-return proxy and next-day execution", "passed": True, "evidence_or_action": "SPY adjusted close; shifted positions"},
    {"gate": "Cash yield and cost sensitivity", "passed": True, "evidence_or_action": "lagged ^IRX; 0–50 bps grid"},
    {"gate": "Serial-dependence-aware uncertainty", "passed": True, "evidence_or_action": "63-day mean-block stationary bootstrap"},
    {"gate": "Calibration and multiple-testing checks", "passed": True, "evidence_or_action": "decile calibration; circular-shift placebo; Holm adjustment"},
    {"gate": "External-market validation executed", "passed": external_markets_completed >= 3, "evidence_or_action": f"{external_markets_completed} markets completed; target at least 3"},
    {"gate": "Genuinely untouched confirmatory sample", "passed": False, "evidence_or_action": "collect future observations or use a never-inspected archival sample"},
    {"gate": "Point-in-time macro data", "passed": False, "evidence_or_action": "replace latest-vintage FRED data with ALFRED/vintage releases"},
    {"gate": "Complete number of tried specifications declared", "passed": DECLARED_TOTAL_TRIALS is not None, "evidence_or_action": "fill DECLARED_TOTAL_TRIALS from all notebooks/manual tests"},
    {"gate": "Independent replication", "passed": False, "evidence_or_action": "second researcher must rebuild results from frozen protocol"},
])
print(publication_gate.to_string(index=False))
save_csv(publication_gate, "publication_gate", index=False)

blocking = publication_gate.loc[~publication_gate["passed"], "gate"].tolist()
print("\nPublication blockers still requiring external evidence:")
for item in blocking:
    print(" -", item)

[saved] /content/outputs/known_trial_registry_20260922_052932_UTC.csv
                                                gate  passed                                                   evidence_or_action
              Notebook runs from clean kernel source    True                     Section G loader restored; no stale output state
                       Causal purged training labels    True                         known_pos purge for regime and 63-day labels
        Leaky/purged comparison uses identical dates    True                          Section F joint masks and equal-n assertion
                     Episode-specific event warnings    True                                 fresh off-to-on transitions required
Investable total-return proxy and next-day execution    True                                SPY adjusted close; shifted positions
                     Cash yield and cost sensitivity    True                                           lagged ^IRX; 0–50 bps grid
                 Ser

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section N — reproducibility manifest and final outputs

In [20]:
# Save the analysis frame only after every publication model has run.
save_csv(df, "daily_series_publication")

packages = {}
for package in ["numpy", "pandas", "scikit-learn", "yfinance", "hmmlearn", "pandas-datareader", "matplotlib"]:
    try:
        packages[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        packages[package] = None

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "created_utc": STAMP,
    "protocol_sha256": PROTOCOL_HASH,
    "python": sys.version,
    "platform": platform.platform(),
    "packages": packages,
    "primary_data_start": str(df.index.min().date()),
    "primary_data_end": str(df.index.max().date()),
    "primary_rows": int(len(df)),
    "random_seeds": {"global": 42, "bootstrap": 20260714, "placebo": 314159},
    "confirmatory_status": "retrospective; not untouched",
    "files": [],
}

# Record artifacts created before the manifest itself.
for path in SAVED_FILES:
    if os.path.exists(path):
        manifest["files"].append({"path": path, "sha256": sha256_file(path), "bytes": os.path.getsize(path)})
manifest_path = save_json(manifest, "reproducibility_manifest")

hr("PUBLICATION AUDIT COMPLETE")
print(f"Protocol: {PROTOCOL_HASH}")
print(f"Manifest: {manifest_path}")
print("The notebook provides a rigorous retrospective analysis. A confirmatory publication claim still requires every failed gate above to be resolved.")

[saved] /content/outputs/daily_series_publication_20260922_052932_UTC.csv
[saved] /content/outputs/reproducibility_manifest_20260922_052932_UTC.json

PUBLICATION AUDIT COMPLETE
Protocol: f9c8178ef9a7b12843c392f528b744338a993d7f537fe799fa9e626f23f67dd6
Manifest: /content/outputs/reproducibility_manifest_20260922_052932_UTC.json
The notebook provides a rigorous retrospective analysis. A confirmatory publication claim still requires every failed gate above to be resolved.


In [21]:
# PUBLICATION OUTPUT LAYER — CELL 1
# Run after analysis_v4_publication(1).ipynb has completed.

from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
)

PUBLICATION_OUT = Path(OUT) if "OUT" in globals() else Path("/content/outputs")
PUBLICATION_OUT.mkdir(parents=True, exist_ok=True)

def save_publication_csv(frame, filename):
    path = PUBLICATION_OUT / filename
    frame.to_csv(path)
    print(f"[saved] {path}")
    return path

def save_publication_figure(fig, filename):
    path = PUBLICATION_OUT / filename
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print(f"[saved] {path}")
    return path

def classification_metrics(prob, target, name, threshold=0.50):
    mask = prob.notna() & target.notna()
    p = prob.loc[mask].astype(float)
    y = target.loc[mask].astype(int)

    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()

    return {
        "model": name,
        "n": len(y),
        "base_rate": y.mean(),
        "ROC_AUC": roc_auc_score(y, p),
        "PR_AUC": average_precision_score(y, p),
        "Brier": brier_score_loss(y, p),
        "LogLoss": log_loss(y, p, labels=[0, 1]),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "FP_rate": fp / (fp + tn) if (fp + tn) else np.nan,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }

required_objects = [
    "df",
    "rule_ma200",
    "evaluate",
]

missing_objects = [x for x in required_objects if x not in globals()]
if missing_objects:
    raise RuntimeError(
        f"Run the complete v4 notebook first. Missing objects: {missing_objects}"
    )

print("Publication output layer initialized.")
print(f"Output directory: {PUBLICATION_OUT}")
print(f"Sample: {df.index.min().date()} to {df.index.max().date()}")
print(f"Rows: {len(df)}")

Publication output layer initialized.
Output directory: /content/outputs
Sample: 1990-01-03 to 2026-09-21
Rows: 9246


In [22]:
# PUBLICATION OUTPUT LAYER — CELL 2
# Clean tables use purged models wherever available.

target_riskoff = df["risk_off"].astype(float)
target_drawdown = df["target_drawdown"]

# Primary purged classification table
purged_rows = [
    classification_metrics(
        df["logit_riskoff_prob_purged"],
        target_riskoff,
        "Logit nowcast (purged)",
    ),
    classification_metrics(
        df["logit_drawdown_prob_purged"],
        target_drawdown,
        "Logit forecast (purged)",
    ),
    classification_metrics(
        df["gbm_drawdown_prob_purged"],
        target_drawdown,
        "GBM forecast (purged)",
    ),
]

purged_classification = pd.DataFrame(purged_rows).set_index("model")
save_publication_csv(
    purged_classification.round(6),
    "publication_table_purged_classification.csv",
)

# Common-date nowcast comparison
nowcast_series = {
    "MA200 rule": rule_ma200.astype(float),
    "HMM filtered": df["hmm_filtered_riskoff"],
    "Logit nowcast (purged)": df["logit_riskoff_prob_purged"],
}

common_nowcast = target_riskoff.notna()
for series in nowcast_series.values():
    common_nowcast &= series.notna()

nowcast_rows = []
for name, series in nowcast_series.items():
    result = classification_metrics(
        series.loc[common_nowcast],
        target_riskoff.loc[common_nowcast],
        name,
    )
    nowcast_rows.append(result)

publication_nowcast = pd.DataFrame(nowcast_rows).set_index("model")
save_publication_csv(
    publication_nowcast.round(6),
    "publication_table_nowcast_common_dates.csv",
)

# Common-date forward forecast comparison
forecast_series = {
    "HMM simulation forecast": df["hmm_drawdown_prob"],
    "Logit forecast (purged)": df["logit_drawdown_prob_purged"],
    "GBM forecast (purged)": df["gbm_drawdown_prob_purged"],
}

common_forecast = target_drawdown.notna()
for series in forecast_series.values():
    common_forecast &= series.notna()

forecast_rows = []
for name, series in forecast_series.items():
    result = classification_metrics(
        series.loc[common_forecast],
        target_drawdown.loc[common_forecast],
        name,
    )
    forecast_rows.append(result)

publication_forecast = pd.DataFrame(forecast_rows).set_index("model")
save_publication_csv(
    publication_forecast.round(6),
    "publication_table_forecast_common_dates.csv",
)

# Export the exact series used by the paper
export_columns = [
    "price",
    "log_ret",
    "bull_bear",
    "risk_off",
    "drawdown",
    "target_drawdown",
    "hmm_filtered_riskoff",
    "hmm_drawdown_prob",
    "logit_riskoff_prob_purged",
    "logit_drawdown_prob_purged",
    "gbm_drawdown_prob_purged",
]

available_columns = [c for c in export_columns if c in df.columns]
publication_series = df[available_columns].copy()
publication_series["ma200_signal"] = rule_ma200.astype(float)

save_publication_csv(
    publication_series.round(8),
    "publication_daily_series.csv",
)

print("\nNOWCAST TABLE")
display(publication_nowcast.round(4))

print("\nFORWARD FORECAST TABLE")
display(publication_forecast.round(4))

[saved] /content/outputs/publication_table_purged_classification.csv
[saved] /content/outputs/publication_table_nowcast_common_dates.csv
[saved] /content/outputs/publication_table_forecast_common_dates.csv
[saved] /content/outputs/publication_daily_series.csv

NOWCAST TABLE


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,n,base_rate,ROC_AUC,PR_AUC,Brier,LogLoss,Precision,Recall,F1,FP_rate,TP,FP,FN,TN
model,,,,,,,,,,,,,,
MA200 rule,6419,0.1263,0.8565,0.3859,0.1715,6.1823,0.4167,0.8940,0.5684,0.1810,725,1015,86,4593
HMM filtered,6419,0.1263,0.8618,0.4007,0.1718,0.7962,0.3457,0.7928,0.4815,0.2170,643,1217,168,4391
Logit nowcast (purged),6419,0.1263,0.8619,0.4361,0.0976,0.3808,0.4863,0.3933,0.4349,0.0601,319,337,492,5271



FORWARD FORECAST TABLE


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,n,base_rate,ROC_AUC,PR_AUC,Brier,LogLoss,Precision,Recall,F1,FP_rate,TP,FP,FN,TN
model,,,,,,,,,,,,,,
HMM simulation forecast,7154,0.1677,0.5862,0.2120,0.1397,0.4545,0.0000,0.0000,0.0000,0.0000,0,0,1200,5954
Logit forecast (purged),7154,0.1677,0.6308,0.3218,0.1496,0.5537,0.3461,0.1583,0.2173,0.0603,190,359,1010,5595
GBM forecast (purged),7154,0.1677,0.5892,0.2123,0.1696,0.8363,0.2144,0.0967,0.1333,0.0714,116,425,1084,5529


In [23]:
# PUBLICATION OUTPUT LAYER — CELL 3
# Figure 1: study design and leakage-safe evaluation pipeline.

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.axis("off")

boxes = [
    (0.02, "Daily S&P 500 prices\n1990–2026"),
    (0.20, "Causal features\nand returns"),
    (0.38, "Ex-post labels\n20% bear rule"),
    (0.56, "Walk-forward models\nHMM / Logit / GBM"),
    (0.74, "Purging and\nbenchmark comparison"),
    (0.91, "Risk metrics and\nstrategy evaluation"),
]

for i, (x, label) in enumerate(boxes):
    ax.text(
        x,
        0.5,
        label,
        ha="center",
        va="center",
        fontsize=11,
        bbox=dict(
            boxstyle="round,pad=0.7",
            facecolor="#e8f1fb",
            edgecolor="#28527a",
            linewidth=1.5,
        ),
        transform=ax.transAxes,
    )

    if i < len(boxes) - 1:
        next_x = boxes[i + 1][0]
        ax.annotate(
            "",
            xy=(next_x - 0.055, 0.5),
            xytext=(x + 0.055, 0.5),
            xycoords=ax.transAxes,
            arrowprops=dict(
                arrowstyle="->",
                lw=1.5,
                color="#555555",
            ),
        )

ax.set_title(
    "Study design and leakage-safe evaluation pipeline",
    fontsize=15,
    weight="bold",
    pad=20,
)

save_publication_figure(fig, "figure_1_study_design.png")

[saved] /content/outputs/figure_1_study_design.png


PosixPath('/content/outputs/figure_1_study_design.png')

In [24]:
# PUBLICATION OUTPUT LAYER — CELL 4
# Figure 2: S&P 500 price, ex-post bear labels, and real-time HMM probability.

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(14, 8),
    sharex=True,
    gridspec_kw={"height_ratios": [2, 1]},
)

dates = df.index
price = df["price"]

ax1.plot(
    dates,
    price,
    color="#1f4e79",
    linewidth=1.1,
    label="S&P 500 adjusted close",
)

# Shade ex-post bear periods
risk = df["risk_off"].fillna(0).astype(int)
start = None

for date, value in risk.items():
    if value == 1 and start is None:
        start = date
    elif value == 0 and start is not None:
        ax1.axvspan(start, date, color="#d62728", alpha=0.15)
        ax2.axvspan(start, date, color="#d62728", alpha=0.15)
        start = None

if start is not None:
    ax1.axvspan(start, dates[-1], color="#d62728", alpha=0.15)
    ax2.axvspan(start, dates[-1], color="#d62728", alpha=0.15)

ax1.set_ylabel("Index level")
ax1.set_title(
    "S&P 500 price and ex-post bear-market dating",
    fontsize=14,
    weight="bold",
)
ax1.legend(loc="upper left")
ax1.grid(alpha=0.25)

ax2.plot(
    dates,
    df["hmm_filtered_riskoff"],
    color="#ff7f0e",
    linewidth=0.9,
    label="Filtered HMM risk-off probability",
)
ax2.axhline(
    0.50,
    color="black",
    linestyle="--",
    linewidth=0.9,
    label="0.50 threshold",
)
ax2.set_ylabel("Probability")
ax2.set_ylim(-0.02, 1.02)
ax2.set_xlabel("Date")
ax2.legend(loc="upper left")
ax2.grid(alpha=0.25)

ax2.xaxis.set_major_locator(mdates.YearLocator(5))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.tight_layout()
save_publication_figure(fig, "figure_2_price_regime_hmm.png")

[saved] /content/outputs/figure_2_price_regime_hmm.png


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PosixPath('/content/outputs/figure_2_price_regime_hmm.png')

In [25]:
# PUBLICATION OUTPUT LAYER — CELL 5
# Figure 3: event-study panels.

events = [
    {
        "name": "2008 Global Financial Crisis",
        "start": "2007-01-01",
        "end": "2009-12-31",
        "onset": "2007-10-10",
        "bottom": "2008-11-20",
    },
    {
        "name": "COVID-19 crash",
        "start": "2019-10-01",
        "end": "2020-06-30",
        "onset": "2020-02-20",
        "bottom": "2020-03-23",
    },
    {
        "name": "2022 bear market",
        "start": "2021-01-01",
        "end": "2022-12-31",
        "onset": "2022-01-04",
        "bottom": "2022-10-12",
    },
]

fig, axes = plt.subplots(
    len(events),
    2,
    figsize=(15, 11),
    gridspec_kw={"width_ratios": [2, 1]},
)

for row, event in enumerate(events):
    mask = (df.index >= event["start"]) & (df.index <= event["end"])
    d = df.loc[mask].copy()

    if len(d) == 0:
        continue

    local_peak = d["price"].cummax()
    local_drawdown = d["price"] / local_peak - 1.0

    ax_price = axes[row, 0]
    ax_prob = axes[row, 1]

    ax_price.plot(
        d.index,
        d["price"],
        color="#1f4e79",
        linewidth=1.1,
    )
    ax_price.set_title(event["name"], fontsize=12, weight="bold")
    ax_price.set_ylabel("Index level")
    ax_price.grid(alpha=0.25)

    ax_price.axvline(
        pd.Timestamp(event["onset"]),
        color="#d62728",
        linestyle="--",
        linewidth=1.2,
        label="Bear onset",
    )
    ax_price.axvline(
        pd.Timestamp(event["bottom"]),
        color="#2ca02c",
        linestyle=":",
        linewidth=1.2,
        label="Bottom",
    )

    ax_dd = ax_price.twinx()
    ax_dd.plot(
        d.index,
        local_drawdown,
        color="#777777",
        linewidth=0.8,
        alpha=0.7,
        label="Drawdown",
    )
    ax_dd.set_ylabel("Drawdown", color="#777777")
    ax_dd.tick_params(axis="y", colors="#777777")

    if row == 0:
        ax_price.legend(loc="upper left", fontsize=8)

    ax_prob.plot(
        d.index,
        d["hmm_filtered_riskoff"],
        color="#ff7f0e",
        linewidth=1.0,
        label="HMM",
    )

    if "logit_riskoff_prob_purged" in d:
        ax_prob.plot(
            d.index,
            d["logit_riskoff_prob_purged"],
            color="#2ca02c",
            linewidth=1.0,
            label="Purged logit",
        )

    ax_prob.axhline(
        0.50,
        color="black",
        linestyle="--",
        linewidth=0.8,
    )
    ax_prob.axvline(
        pd.Timestamp(event["onset"]),
        color="#d62728",
        linestyle="--",
        linewidth=1.2,
    )
    ax_prob.set_ylim(-0.02, 1.02)
    ax_prob.set_ylabel("Risk-off probability")
    ax_prob.grid(alpha=0.25)

    if row == 0:
        ax_prob.legend(loc="upper left", fontsize=8)

    for axis in [ax_price, ax_prob]:
        axis.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
        axis.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

fig.suptitle(
    "Event studies: gradual crises and abrupt shocks",
    fontsize=16,
    weight="bold",
    y=0.995,
)

fig.tight_layout()
save_publication_figure(fig, "figure_3_event_studies.png")

[saved] /content/outputs/figure_3_event_studies.png


PosixPath('/content/outputs/figure_3_event_studies.png')

In [26]:
# PUBLICATION OUTPUT LAYER — CELL 6
# Figure 4: threshold sensitivity using purged forecast outputs.

def threshold_curve(prob, target, model_name):
    mask = prob.notna() & target.notna()
    p = prob.loc[mask].astype(float)
    y = target.loc[mask].astype(int)

    rows = []

    for threshold in np.arange(0.10, 0.91, 0.05):
        pred = (p >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            y,
            pred,
            labels=[0, 1],
        ).ravel()

        rows.append(
            {
                "model": model_name,
                "threshold": threshold,
                "precision": precision_score(y, pred, zero_division=0),
                "recall": recall_score(y, pred, zero_division=0),
                "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
            }
        )

    return pd.DataFrame(rows)

threshold_results = pd.concat(
    [
        threshold_curve(
            df["logit_drawdown_prob_purged"],
            target_drawdown,
            "Purged logistic",
        ),
        threshold_curve(
            df["gbm_drawdown_prob_purged"],
            target_drawdown,
            "Purged gradient boosting",
        ),
    ],
    ignore_index=True,
)

save_publication_csv(
    threshold_results.round(6),
    "publication_table_purged_thresholds.csv",
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True)

metrics = [
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("fpr", "False-positive rate"),
]

for ax, (column, title) in zip(axes, metrics):
    for model_name, group in threshold_results.groupby("model"):
        ax.plot(
            group["threshold"],
            group[column],
            marker="o",
            linewidth=1.5,
            label=model_name,
        )

    ax.set_title(title)
    ax.set_xlabel("Probability threshold")
    ax.set_ylabel(title)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)

axes[0].legend(fontsize=8, loc="best")

fig.suptitle(
    "Threshold sensitivity for leakage-corrected drawdown forecasts",
    fontsize=15,
    weight="bold",
)

fig.tight_layout()
save_publication_figure(fig, "figure_4_purged_threshold_sensitivity.png")

[saved] /content/outputs/publication_table_purged_thresholds.csv


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[saved] /content/outputs/figure_4_purged_threshold_sensitivity.png


PosixPath('/content/outputs/figure_4_purged_threshold_sensitivity.png')

In [27]:
# PUBLICATION OUTPUT LAYER — CELL 7
# Figure 5: cumulative wealth and drawdowns.
# Uses the common retrospective strategy sample created in Section H.

if "strategy_daily_returns" not in globals():
    raise RuntimeError(
        "strategy_daily_returns is missing. Run the complete v4 notebook first."
    )

strategy_names = [
    "Buy & hold",
    "MA200",
    "Price-only Logit (purged)",
    "HMM filtered",
]

available_strategies = [
    name for name in strategy_names if name in strategy_daily_returns
]

if len(available_strategies) < 2:
    raise RuntimeError(
        "Not enough strategy return series were found for Figure 5."
    )

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(14, 8),
    sharex=True,
)

colors = {
    "Buy & hold": "#1f4e79",
    "MA200": "#2ca02c",
    "Price-only Logit (purged)": "#9467bd",
    "HMM filtered": "#ff7f0e",
}

for name in available_strategies:
    returns = strategy_daily_returns[name].dropna()
    wealth = (1.0 + returns).cumprod()
    drawdown = wealth / wealth.cummax() - 1.0

    ax1.plot(
        wealth.index,
        wealth,
        linewidth=1.5,
        label=name,
        color=colors.get(name),
    )

    ax2.plot(
        drawdown.index,
        drawdown,
        linewidth=1.2,
        label=name,
        color=colors.get(name),
    )

ax1.set_ylabel("Growth of $1")
ax1.set_title(
    "Cumulative wealth, common retrospective sample",
    fontsize=13,
    weight="bold",
)
ax1.legend(loc="upper left")
ax1.grid(alpha=0.25)

ax2.set_ylabel("Drawdown")
ax2.set_xlabel("Date")
ax2.set_title("Underwater drawdown")
ax2.axhline(0, color="black", linewidth=0.8)
ax2.grid(alpha=0.25)

ax2.xaxis.set_major_locator(mdates.YearLocator(2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.tight_layout()
save_publication_figure(fig, "figure_5_strategy_equity_curves.png")

[saved] /content/outputs/figure_5_strategy_equity_curves.png


PosixPath('/content/outputs/figure_5_strategy_equity_curves.png')

In [28]:
# PUBLICATION OUTPUT LAYER — CELL 8
# Final checks before putting numbers into the manuscript.

required_figures = [
    "figure_1_study_design.png",
    "figure_2_price_regime_hmm.png",
    "figure_3_event_studies.png",
    "figure_4_purged_threshold_sensitivity.png",
    "figure_5_strategy_equity_curves.png",
]

missing_figures = [
    name for name in required_figures
    if not (PUBLICATION_OUT / name).exists()
]

if missing_figures:
    raise RuntimeError(f"Missing publication figures: {missing_figures}")

required_columns = [
    "hmm_filtered_riskoff",
    "logit_riskoff_prob_purged",
    "logit_drawdown_prob_purged",
    "gbm_drawdown_prob_purged",
    "target_drawdown",
]

missing_columns = [
    name for name in required_columns
    if name not in df.columns
]

if missing_columns:
    raise RuntimeError(f"Missing required dataframe columns: {missing_columns}")

assert df["hmm_filtered_riskoff"].dropna().between(0, 1).all()
assert df["logit_riskoff_prob_purged"].dropna().between(0, 1).all()
assert df["logit_drawdown_prob_purged"].dropna().between(0, 1).all()
assert df["gbm_drawdown_prob_purged"].dropna().between(0, 1).all()

print("Publication QA passed.")
print("\nGenerated figures:")
for name in required_figures:
    print(f"  - {PUBLICATION_OUT / name}")

print("\nGenerated tables:")
for path in sorted(PUBLICATION_OUT.glob("publication_*.csv")):
    print(f"  - {path}")

Publication QA passed.

Generated figures:
  - /content/outputs/figure_1_study_design.png
  - /content/outputs/figure_2_price_regime_hmm.png
  - /content/outputs/figure_3_event_studies.png
  - /content/outputs/figure_4_purged_threshold_sensitivity.png
  - /content/outputs/figure_5_strategy_equity_curves.png

Generated tables:
  - /content/outputs/publication_daily_series.csv
  - /content/outputs/publication_gate_20260922_052932_UTC.csv
  - /content/outputs/publication_investable_strategy_20260922_052932_UTC.csv
  - /content/outputs/publication_table_forecast_common_dates.csv
  - /content/outputs/publication_table_nowcast_common_dates.csv
  - /content/outputs/publication_table_purged_classification.csv
  - /content/outputs/publication_table_purged_thresholds.csv
